
# ModelGuard – AI Lifecycle Management & Governance Agent

## Notebook 05 – LLM Agent Architecture, System Integration and Implementation

## Notebook Overview

This notebook focuses on developing and integrating the ModelGuard intelligent agent into a unified AI lifecycle management and governance system.

The agent uses a locally hosted Large Language Model (LLM) through Ollama to understand user requests, retrieve relevant information, plan tasks, execute predefined tools, and generate evidence-based responses.

The implementation connects the agent with the previously developed components, including the customer churn prediction model, MLflow experiment tracking, model monitoring, drift detection, and AI governance framework.

The notebook also focuses on integrating memory, reasoning, planning, ReAct execution, response validation, and audit logging to ensure that the agent operates in a structured and traceable way.

Finally, a FastAPI interface will be developed to enable interaction with the integrated system. The completed prototype will be evaluated through functional, integration, validation, and end-to-end testing.

## Main Objectives

- Integrate a locally hosted LLM using Ollama.
- Design and implement the ModelGuard Agent architecture.
- Develop reasoning, planning, and task execution capabilities.
- Implement a ReAct execution workflow.
- Connect the agent to monitoring, MLflow, and AI governance components.
- Integrate the trained XGBoost model for customer churn prediction.
- Implement short-term and persistent memory.
- Develop structured response generation and evidence validation.
- Implement audit logging and execution tracking.
- Handle tool execution errors and incomplete evidence.
- Develop a FastAPI interface for user interaction.
- Evaluate the integrated system through functional and end-to-end testing.
- Assess the system's limitations and identify opportunities for future improvement.


### Check Ollama Connection and Available Models

Checks whether the Ollama service is running locally and retrieves the list of installed language models. It also confirms that the required model is available for integration with the ModelGuard Agent.

In [1]:
import requests

response = requests.get(
    "http://localhost:11434/api/tags",
    timeout=10
)

response.raise_for_status()
models = response.json().get("models", [])

print("Ollama Service: Running")
print("\nAvailable Models:")

if models:
    for model in models:
        print(f"- {model['name']}")
else:
    print("No models installed yet.")

Ollama Service: Running

Available Models:
- qwen2.5:1.5b


### Local LLM Integration

This section connects the locally hosted Qwen model to the ModelGuard environment through the Ollama API.

The connection allows Python to send requests to the language model and receive responses for further processing.

In [2]:
import requests
import json

OLLAMA_URL = "http://localhost:11434/api/generate"
MODEL_NAME = "qwen2.5:1.5b"

test_prompt = """
You are the ModelGuard AI assistant.

A monitoring report shows:
- MonthlyCharges: Data drift detected.
- Contract: Data drift detected.
- Model Recall: 74.6%.

Explain what can be concluded from these results.

Rules:
- Use only the information provided.
- Do not assume that model performance has decreased.
- Do not recommend retraining without sufficient evidence.
- Clearly distinguish detected facts from possible explanations.
"""

payload = {
    "model": MODEL_NAME,
    "prompt": test_prompt,
    "stream": False,
    "options": {
        "temperature": 0
    }
}

response = requests.post(
    OLLAMA_URL,
    json=payload,
    timeout=180
)

response.raise_for_status()

result = response.json()

print("Model:", result["model"])
print("\nResponse:")
print(result["response"])

Model: qwen2.5:1.5b

Response:
Based on the monitoring report provided, the following conclusions can be drawn:

1. **MonthlyCharges: Data drift detected**
   - This indicates that there has been a change in the distribution of the data for the "MonthlyCharges" metric over time. This could be due to various factors such as changes in customer behavior, economic conditions, or other external influences that affect the billing cycle.

2. **Contract: Data drift detected**
   - This suggests that the data distribution for the "Contract" metric has also changed over time. Similar to "MonthlyCharges," this could be due to shifts in customer contracts, changes in pricing structures, or other external factors impacting the contract terms.

3. **Model Recall: 74.6%**
   - The model's recall rate is 74.6%, which is a good performance metric. Recall measures the proportion of actual positive cases that are correctly identified by the model. A recall rate of 74.6% indicates that the model is perfo

## ModelGuard Agent Architecture

The ModelGuard Agent is designed as a modular system that connects a locally hosted language model with the existing monitoring, MLflow, and governance components.

The architecture consists of an Agent Core, a Tool Execution Layer, ModelGuard Services, and a Logging mechanism.

The Agent Core handles reasoning, planning, memory, and ReAct-based execution. The Tool Execution Layer connects the agent to predefined Python functions that retrieve and process system information.

The language model is responsible for understanding user requests and interpreting tool results, while statistical calculations and governance checks are performed by deterministic Python functions.

The system also considers the Model Context Protocol (MCP) as a potential future integration approach. However, MCP is not implemented in this project.

## Agent Configuration and Core Setup

This section defines the core configuration of the ModelGuard Agent.

It specifies the language model, Ollama API connection, generation settings, and execution limits. These settings will be shared across the agent components to maintain consistency during execution.

In [3]:
import requests
import json
import time
from datetime import datetime
from pathlib import Path

# Agent Configuration
AGENT_NAME = "ModelGuard Agent"

MODEL_NAME = "qwen2.5:1.5b"
OLLAMA_URL = "http://localhost:11434"
GENERATE_URL = f"{OLLAMA_URL}/api/generate"
TAGS_URL = f"{OLLAMA_URL}/api/tags"

# Model Generation Settings
MODEL_TEMPERATURE = 0
MODEL_TIMEOUT = 180

# Agent Execution Settings
MAX_ITERATIONS = 5

# HTTP Session
session = requests.Session()

# Verify Ollama Connection
response = session.get(TAGS_URL, timeout=10)
response.raise_for_status()

available_models = [
    model["name"]
    for model in response.json().get("models", [])
]

if MODEL_NAME not in available_models:
    raise ValueError(
        f"Model {MODEL_NAME} is not installed."
    )

print("Agent configuration completed successfully.")
print(f"Agent Name: {AGENT_NAME}")
print(f"Language Model: {MODEL_NAME}")
print(f"Temperature: {MODEL_TEMPERATURE}")
print(f"Maximum Iterations: {MAX_ITERATIONS}")
print("Ollama Connection: Verified")

Agent configuration completed successfully.
Agent Name: ModelGuard Agent
Language Model: qwen2.5:1.5b
Temperature: 0
Maximum Iterations: 5
Ollama Connection: Verified


## ModelGuard Tool Integration Layer

This section implements a centralized tool integration layer for ModelGuard.

The predefined Python tools provide the agent with access to model monitoring, MLflow experiments, governance records, risk assessments, and system information.

A centralized tool registry is used to organize the available functions. An automated integration test will validate tool availability, execution, and output structure before connecting them to the agent.

### Tool Registry and Function Implementation

This section implements the ModelGuard tools and registers them in a centralized tool registry.

The tools provide access to monitoring reports, model information, MLflow experiments, and governance records. Each tool retrieves or processes existing project evidence and returns structured results.

The registry provides a single interface for managing and executing the available tools.

In [4]:

import json
import pandas as pd
import numpy as np
import mlflow

from scipy.stats import ks_2samp, chi2_contingency
from mlflow.tracking import MlflowClient
from pathlib import Path

# --------------------------------------------------
# 1. Project Paths
# --------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MONITORING_DIR = PROJECT_ROOT / "monitoring"
GOVERNANCE_DIR = PROJECT_ROOT / "governance"

TRACKING_URI = f"sqlite:///{(PROJECT_ROOT / 'mlflow.db').as_posix()}"

# --------------------------------------------------
# 2. Helper Functions
# --------------------------------------------------

def load_json(path):
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.name}")

    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def load_csv(path):
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path.name}")

    return pd.read_csv(path)


# --------------------------------------------------
# 3. Model Monitoring Tools
# --------------------------------------------------

def get_model_information():
    results = load_json(ARTIFACTS_DIR / "model_results.json")

    return {
        "model_results": results.get("xgboost"),
        "baseline_model": results.get("logistic_regression"),
        "experimental_model": results.get("xgboost_undersampling")
    }


def get_performance_baseline():
    return load_json(MONITORING_DIR / "performance_baseline.json")


def get_batch_performance():
    df = load_csv(MONITORING_DIR / "batch_performance_report.csv")

    return df.to_dict(orient="records")


def get_drift_analysis():
    reference = load_csv(MONITORING_DIR / "reference_data.csv")
    comparison = load_csv(MONITORING_DIR / "comparison_data.csv")

    numerical_features = [
        "tenure", "MonthlyCharges", "TotalCharges"
    ]

    categorical_features = [
        column for column in reference.columns
        if column not in numerical_features
    ]

    alpha = 0.05
    adjusted_alpha = alpha / len(reference.columns)

    results = []

    for feature in numerical_features:
        test = ks_2samp(
            reference[feature].dropna(),
            comparison[feature].dropna()
        )

        results.append({
            "feature": feature,
            "test": "Kolmogorov-Smirnov",
            "statistic": float(test.statistic),
            "p_value": float(test.pvalue),
            "adjusted_alpha": adjusted_alpha,
            "drift_detected": bool(test.pvalue < adjusted_alpha)
        })

    for feature in categorical_features:
        categories = sorted(
            set(reference[feature].dropna().unique()) |
            set(comparison[feature].dropna().unique()),
            key=str
        )

        table = pd.DataFrame({
            "Reference": [
                (reference[feature] == category).sum()
                for category in categories
            ],
            "Comparison": [
                (comparison[feature] == category).sum()
                for category in categories
            ]
        }, index=categories).T

        chi2, p_value, _, _ = chi2_contingency(table)

        results.append({
            "feature": feature,
            "test": "Chi-Square",
            "statistic": float(chi2),
            "p_value": float(p_value),
            "adjusted_alpha": adjusted_alpha,
            "drift_detected": bool(p_value < adjusted_alpha)
        })

    return {
        "analysis_type": "Reference vs Comparison Data",
        "number_of_features": len(results),
        "drifted_features": [
            item["feature"] for item in results
            if item["drift_detected"]
        ],
        "results": results
    }


# --------------------------------------------------
# 4. Governance and Risk Tools
# --------------------------------------------------

def get_system_profile():
    return load_json(GOVERNANCE_DIR / "ai_system_profile.json")


def get_risk_register():
    return load_json(GOVERNANCE_DIR / "risk_register.json")


def get_impact_assessment():
    return load_json(GOVERNANCE_DIR / "impact_assessment.json")


def get_incident_management():
    return load_json(GOVERNANCE_DIR / "incident_management.json")


def get_governance_controls():
    files = {
        "lifecycle_controls": "lifecycle_controls.json",
        "ai_management_structure": "ai_management_structure.json",
        "framework_assessment": "framework_assessment.json",
        "responsible_ai_principles": "responsible_ai_principles.json"
    }

    return {
        name: load_json(GOVERNANCE_DIR / filename)
        for name, filename in files.items()
    }


# --------------------------------------------------
# 5. MLflow Tools
# --------------------------------------------------

def get_mlflow_client():
    mlflow.set_tracking_uri(TRACKING_URI)
    return MlflowClient(tracking_uri=TRACKING_URI)


def get_mlflow_experiments():
    client = get_mlflow_client()
    experiments = client.search_experiments()

    return [
        {
            "experiment_id": experiment.experiment_id,
            "name": experiment.name,
            "lifecycle_stage": experiment.lifecycle_stage
        }
        for experiment in experiments
    ]


def get_mlflow_run_details(run_id=None):
    client = get_mlflow_client()

    experiment = client.get_experiment_by_name(
        "ModelGuard_Churn_Prediction"
    )

    if experiment is None:
        raise ValueError("ModelGuard MLflow experiment not found.")

    if run_id is not None:
        run = client.get_run(run_id)
        return {
            "run_id": run.info.run_id,
            "status": run.info.status,
            "metrics": run.data.metrics,
            "params": run.data.params,
            "tags": run.data.tags
        }

    runs = client.search_runs(
        experiment_ids=[experiment.experiment_id],
        max_results=100
    )

    return [
        {
            "run_id": run.info.run_id,
            "status": run.info.status,
            "metrics": run.data.metrics,
            "params": run.data.params,
            "tags": run.data.tags
        }
        for run in runs
    ]


# --------------------------------------------------
# 6. System Management Tool
# --------------------------------------------------

def get_system_status():
    required_files = {
        "Model Results": ARTIFACTS_DIR / "model_results.json",
        "Performance Baseline": MONITORING_DIR / "performance_baseline.json",
        "Reference Data": MONITORING_DIR / "reference_data.csv",
        "Comparison Data": MONITORING_DIR / "comparison_data.csv",
        "Batch Performance": MONITORING_DIR / "batch_performance_report.csv",
        "System Profile": GOVERNANCE_DIR / "ai_system_profile.json",
        "Risk Register": GOVERNANCE_DIR / "risk_register.json",
        "Impact Assessment": GOVERNANCE_DIR / "impact_assessment.json",
        "Incident Management": GOVERNANCE_DIR / "incident_management.json",
        "MLflow Database": PROJECT_ROOT / "mlflow.db"
    }

    status = {
        name: "Available" if path.exists() else "Missing"
        for name, path in required_files.items()
    }

    return {
        "components": status,
        "available": sum(value == "Available" for value in status.values()),
        "missing": sum(value == "Missing" for value in status.values())
    }


# --------------------------------------------------
# 7. Centralized Tool Registry
# --------------------------------------------------

TOOL_REGISTRY = {
    "get_model_information": get_model_information,
    "get_performance_baseline": get_performance_baseline,
    "get_batch_performance": get_batch_performance,
    "get_drift_analysis": get_drift_analysis,
    "get_system_profile": get_system_profile,
    "get_risk_register": get_risk_register,
    "get_impact_assessment": get_impact_assessment,
    "get_incident_management": get_incident_management,
    "get_governance_controls": get_governance_controls,
    "get_mlflow_experiments": get_mlflow_experiments,
    "get_mlflow_run_details": get_mlflow_run_details,
    "get_system_status": get_system_status
}

print("ModelGuard Tool Registry created successfully.")
print(f"Registered Tools: {len(TOOL_REGISTRY)}")

for tool_name in TOOL_REGISTRY:
    print(f"- {tool_name}")

ModelGuard Tool Registry created successfully.
Registered Tools: 12
- get_model_information
- get_performance_baseline
- get_batch_performance
- get_drift_analysis
- get_system_profile
- get_risk_register
- get_impact_assessment
- get_incident_management
- get_governance_controls
- get_mlflow_experiments
- get_mlflow_run_details
- get_system_status


### Automated Tool Integration Testing

This section tests all registered ModelGuard tools to verify that they can access their data sources, execute successfully, and return valid outputs.

The test records the execution status of each tool and identifies any errors that may affect the agent integration.

In [5]:

import json
import pandas as pd
import time

# Store integration test results
tool_test_results = []

print("Starting ModelGuard Tool Integration Tests...\n")

for tool_name, tool_function in TOOL_REGISTRY.items():

    start_time = time.perf_counter()

    try:
        # Execute the tool
        result = tool_function()

        # Validate that the result is serializable
        json.dumps(result, default=str)

        if result is None:
            raise ValueError("Tool returned an empty result.")

        execution_time = time.perf_counter() - start_time

        tool_test_results.append({
            "Tool": tool_name,
            "Status": "Passed",
            "Execution_Time_Seconds": round(execution_time, 4),
            "Result_Type": type(result).__name__,
            "Error": None
        })

    except Exception as error:

        execution_time = time.perf_counter() - start_time

        tool_test_results.append({
            "Tool": tool_name,
            "Status": "Failed",
            "Execution_Time_Seconds": round(execution_time, 4),
            "Result_Type": None,
            "Error": str(error)
        })

# Create test report
tool_test_report = pd.DataFrame(tool_test_results)

display(tool_test_report)

# Summary
passed = (tool_test_report["Status"] == "Passed").sum()
failed = (tool_test_report["Status"] == "Failed").sum()

print("\nIntegration Test Summary")
print("------------------------")
print(f"Total Tools: {len(TOOL_REGISTRY)}")
print(f"Passed: {passed}")
print(f"Failed: {failed}")
print(f"Success Rate: {(passed / len(TOOL_REGISTRY)) * 100:.2f}%")


Starting ModelGuard Tool Integration Tests...



,Tool,Status,Execution_Time_Seconds,Result_Type,Error
0,get_model_information,Passed,0.0078,dict,None
1,get_performance_baseline,Passed,0.0066,dict,None
2,get_batch_performance,Passed,0.2144,list,None
3,get_drift_analysis,Passed,0.3847,dict,None
4,get_system_profile,Passed,0.0325,dict,None
5,get_risk_register,Passed,0.0057,list,None
6,get_impact_assessment,Passed,0.0137,list,None
7,get_incident_management,Passed,0.0234,list,None
8,get_governance_controls,Passed,0.0397,dict,None
9,get_mlflow_experiments,Passed,11.3758,list,None



Integration Test Summary
------------------------
Total Tools: 12
Passed: 12
Failed: 0
Success Rate: 100.00%


### Tool Integration Test Report

The integration tests were completed successfully. All twelve registered tools executed without errors and returned valid structured outputs.

The test results are saved to support project documentation and future system validation.

In [6]:
# Save Tool Integration Test Report

REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

csv_path = REPORTS_DIR / "tool_integration_test_report.csv"
json_path = REPORTS_DIR / "tool_integration_test_report.json"

tool_test_report.to_csv(csv_path, index=False)

tool_test_report.to_json(
    json_path,
    orient="records",
    indent=4
)

print("Tool integration reports saved successfully.")
print(f"- {csv_path}")
print(f"- {json_path}")

Tool integration reports saved successfully.
- C:\Users\User\ModelGuard\reports\tool_integration_test_report.csv
- C:\Users\User\ModelGuard\reports\tool_integration_test_report.json


## Agent Tool Calling

This section connects the language model to the registered ModelGuard tools.

The agent receives a user request, identifies the appropriate tool, and returns a structured tool call. Python validates the requested function and executes it through the centralized tool registry.

The tool result is then returned to the language model to support an evidence-based response.

### Tool Descriptions

Each registered tool is assigned a clear description to help the language model identify which function should be used for a given request.

In [7]:

# Tool descriptions provided to the LLM

TOOL_DESCRIPTIONS = {
    "get_model_information": "Retrieve model information, configurations, and evaluation results.",
    "get_performance_baseline": "Retrieve the baseline performance metrics of the churn prediction model.",
    "get_batch_performance": "Compare model performance across different evaluation batches.",
    "get_drift_analysis": "Analyze statistical data drift between reference and comparison datasets.",
    "get_system_profile": "Retrieve the AI system profile, purpose, and deployment status.",
    "get_risk_register": "Retrieve identified AI system risks, scores, and mitigation measures.",
    "get_impact_assessment": "Retrieve the impact assessment for affected stakeholders.",
    "get_incident_management": "Retrieve incident scenarios, responses, and responsibilities.",
    "get_governance_controls": "Retrieve AI governance, lifecycle, and framework controls.",
    "get_mlflow_experiments": "Retrieve registered MLflow experiments.",
    "get_mlflow_run_details": "Retrieve MLflow run metrics, parameters, and tags.",
    "get_system_status": "Check the availability of ModelGuard system components."
}

# Verify descriptions against registered tools
assert set(TOOL_DESCRIPTIONS) == set(TOOL_REGISTRY)

print("Tool descriptions configured successfully.")
print(f"Described Tools: {len(TOOL_DESCRIPTIONS)}")


Tool descriptions configured successfully.
Described Tools: 12


### LLM-Based Tool Selection

The language model is instructed to select the most relevant tool based on the user's request.


It must return a structured JSON response containing the selected tool name. The selected function is not executed at this stage; execution will only take place after validation.

In [8]:

def select_tool(user_request):

    tool_list = "\n".join(
        f"- {name}: {description}"
        for name, description in TOOL_DESCRIPTIONS.items()
    )

    prompt = f"""
You are the tool selection component of ModelGuard.

Select the single most appropriate tool for the user's request.

Available tools:
{tool_list}

User request:
{user_request}

Rules:
- Select only one tool from the available list.
- Do not invent tool names.
- If no tool is relevant, return null.
- Return only valid JSON.
- Do not execute the tool.

Required JSON format:
{{
    "tool_name": "exact_tool_name_or_null"
}}
"""

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": 0
        }
    }

    response = session.post(
        GENERATE_URL,
        json=payload,
        timeout=MODEL_TIMEOUT
    )

    response.raise_for_status()

    model_output = response.json()["response"]
    selection = json.loads(model_output)

    tool_name = selection.get("tool_name")

    if tool_name is not None and tool_name not in TOOL_REGISTRY:
        raise ValueError(f"Invalid tool selected: {tool_name}")

    return selection


# Test tool selection
test_request = "Check whether the churn model has experienced data drift."

selection = select_tool(test_request)

print("User Request:", test_request)
print("Selected Tool:", selection["tool_name"])


User Request: Check whether the churn model has experienced data drift.
Selected Tool: get_drift_analysis


### Tool Execution and Result Retrieval

This section implements the tool execution mechanism.

The selected tool is validated against the registered tool registry before execution. The function result is then returned in a structured format for further processing by the language model.

In [9]:

def execute_tool(tool_name):

    if tool_name not in TOOL_REGISTRY:
        raise ValueError(f"Unknown tool: {tool_name}")

    start_time = time.perf_counter()

    try:
        result = TOOL_REGISTRY[tool_name]()

        execution_time = time.perf_counter() - start_time

        return {
            "tool_name": tool_name,
            "status": "success",
            "execution_time": round(execution_time, 4),
            "result": result
        }

    except Exception as error:
        return {
            "tool_name": tool_name,
            "status": "error",
            "execution_time": round(time.perf_counter() - start_time, 4),
            "error": str(error)
        }


# Execute the selected tool
tool_result = execute_tool(selection["tool_name"])

print("Tool Execution Status:", tool_result["status"])
print("Tool Name:", tool_result["tool_name"])
print("Execution Time:", tool_result["execution_time"], "seconds")

if tool_result["status"] == "success":
    print("\nTool Result:")
    print(json.dumps(tool_result["result"], indent=2, default=str))
else:
    print("Error:", tool_result["error"])


Tool Execution Status: success
Tool Name: get_drift_analysis
Execution Time: 0.2582 seconds

Tool Result:
{
  "analysis_type": "Reference vs Comparison Data",
  "number_of_features": 19,
  "drifted_features": [],
  "results": [
    {
      "feature": "tenure",
      "test": "Kolmogorov-Smirnov",
      "statistic": 0.021175676523429665,
      "p_value": 0.6834995512009212,
      "adjusted_alpha": 0.002631578947368421,
      "drift_detected": false
    },
    {
      "feature": "MonthlyCharges",
      "test": "Kolmogorov-Smirnov",
      "statistic": 0.032085434852221595,
      "p_value": 0.19115443508495597,
      "adjusted_alpha": 0.002631578947368421,
      "drift_detected": false
    },
    {
      "feature": "TotalCharges",
      "test": "Kolmogorov-Smirnov",
      "statistic": 0.029577343075462226,
      "p_value": 0.2709525960188033,
      "adjusted_alpha": 0.002631578947368421,
      "drift_detected": false
    },
    {
      "feature": "gender",
      "test": "Chi-Square",
      

### Evidence-Based Response Generation

In this section, the tool execution result is passed back to the language model.

The model interprets the retrieved evidence and generates a response based on the actual monitoring results. It must distinguish between observed findings and unsupported assumptions.

In [10]:

def generate_tool_response(user_request, tool_result):

    if tool_result["status"] != "success":
        return "The requested tool could not be executed."

    evidence = json.dumps(
        tool_result["result"],
        indent=2,
        default=str
    )

    prompt = f"""
You are the ModelGuard AI assistant.

User request:
{user_request}

Tool executed:
{tool_result["tool_name"]}

Actual tool results:
{evidence}

Strict analytical rules:

1. Use only the evidence provided by the tool.
2. Report the exact number of analyzed features.
3. Identify the exact features where drift was detected.
4. Never contradict the returned drift_detected values.
5. If drifted_features is empty, explicitly state that
   no features showed statistically significant drift.
6. Do not claim that the model is stable or unaffected.
7. Data drift detection does not prove model performance
   degradation or stability.
8. Clearly distinguish statistical findings from recommendations.
9. Mention that the comparison uses reference and test data,
   not live production data.

Provide a concise and scientifically accurate response.
"""

    payload = {
        "model": MODEL_NAME,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0
        }
    }

    response = session.post(
        GENERATE_URL,
        json=payload,
        timeout=MODEL_TIMEOUT
    )

    response.raise_for_status()

    return response.json()["response"]


# Generate evidence-based response
agent_response = generate_tool_response(
    test_request,
    tool_result
)

print("ModelGuard Agent Response:\n")
print(agent_response)


ModelGuard Agent Response:

The churn model has not experienced data drift based on the analysis provided. The tool evaluated 19 features, and none of them showed statistically significant drift. The features that were tested include tenure, MonthlyCharges, TotalCharges, gender, SeniorCitizen, Partner, Dependents, PhoneService, MultipleLines, InternetService, OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV, StreamingMovies, Contract, and PaperlessBilling. All features were found to be stable, with no features showing drift detected.


## Reasoning and Planning

The Reasoning and Planning component analyzes user requests and converts them into structured execution plans.

Reasoning identifies the user's objective and the information required, while Planning selects and organizes the appropriate tools into an ordered sequence of actions.

The generated plan is validated before being passed to the ReAct Controller for execution.

### Prepare Available Tools

In [11]:

PLANNING_TOOLS = {
    name: description
    for name, description in TOOL_DESCRIPTIONS.items()
    if name in TOOL_REGISTRY
}

print("Available planning tools:", len(PLANNING_TOOLS))


Available planning tools: 12


### Implement Reasoning and Planning

In [12]:

def reason_and_plan(user_request):

    prompt = f"""
You are the reasoning and planning component of ModelGuard.

Analyze the user request and create a structured execution plan.

User request:
{user_request}

Available tools:
{json.dumps(PLANNING_TOOLS, ensure_ascii=False)}

Return valid JSON only, using this structure:

{{
    "reasoning": {{
        "objective": "Main purpose of the request",
        "required_information": [
            "Information needed to answer the request"
        ]
    }},
    "plan": [
        {{
            "step": 1,
            "tool_name": "exact_tool_name",
            "purpose": "Purpose of this tool"
        }}
    ]
}}

Rules:
- Select only tools from the available tools list.
- Select the minimum number of tools needed to answer the request.
- Use only directly relevant tools.
- Do not retrieve additional information unless necessary.
- Do not include MLflow tools unless the user asks about experiments or model tracking.
- Organize steps in a logical order.
- Maximum 5 tools per plan.
- Do not execute tools.
- Do not invent tool names.
- If no tool is required, return an empty plan.
- Do not include private chain-of-thought.
"""

    try:
        response = session.post(
            GENERATE_URL,
            json={
                "model": MODEL_NAME,
                "prompt": prompt,
                "stream": False,
                "format": "json",
                "options": {"temperature": MODEL_TEMPERATURE}
            },
            timeout=MODEL_TIMEOUT
        )

        response.raise_for_status()
        result = json.loads(response.json()["response"])

        reasoning = result.get("reasoning", {})
        plan = result.get("plan", [])

        if not isinstance(reasoning, dict) or not isinstance(plan, list):
            raise ValueError("Invalid reasoning or plan structure.")

        # Validate and normalize the execution plan.
        validated_plan = []
        seen_tools = set()

        for step in plan:
            tool_name = step.get("tool_name")

            if tool_name not in PLANNING_TOOLS:
                continue

            if tool_name in seen_tools:
                continue

            seen_tools.add(tool_name)

            validated_plan.append({
                "step": len(validated_plan) + 1,
                "tool_name": tool_name,
                "purpose": str(step.get("purpose", ""))
            })
        
        # Limit the plan to a maximum of five tools.
        validated_plan = validated_plan[:5]
        
        return {
            "status": "success",
            "reasoning": reasoning,
            "plan": validated_plan
        }

    except Exception as error:
        return {
            "status": "error",
            "error": str(error),
            "reasoning": {},
            "plan": []
        }


print("Reasoning and Planning component created successfully.")


Reasoning and Planning component created successfully.



### Reasoning and Planning Testing

This test evaluates whether the agent can understand a user request, identify the required information, and generate an ordered plan using the available tools.


In [13]:

planning_request = """
Evaluate the current condition of the churn prediction model.
Check its performance, data drift, and governance risks.
"""

planning_result = reason_and_plan(planning_request)

print("Status:", planning_result["status"])

print("\nReasoning:")
print(json.dumps(
    planning_result["reasoning"],
    indent=4,
    ensure_ascii=False
))

print("\nExecution Plan:")
for step in planning_result["plan"]:
    print(step)


Status: success

Reasoning:
{
    "objective": "Evaluate the current condition of the churn prediction model, including performance, data drift, and governance risks.",
    "required_information": [
        "Retrieve model information, configurations, and evaluation results.",
        "Retrieve baseline performance metrics of the churn prediction model.",
        "Analyze statistical data drift between reference and comparison datasets.",
        "Retrieve identified AI system risks, scores, and mitigation measures.",
        "Retrieve impact assessment for affected stakeholders."
    ]
}

Execution Plan:
{'step': 1, 'tool_name': 'get_model_information', 'purpose': 'Retrieve model information, configurations, and evaluation results.'}
{'step': 2, 'tool_name': 'get_performance_baseline', 'purpose': 'Retrieve baseline performance metrics of the churn prediction model.'}
{'step': 3, 'tool_name': 'get_drift_analysis', 'purpose': 'Analyze statistical data drift between reference and compari


## ReAct Controller

The ReAct Controller manages the execution of the plan generated by the Reasoning and Planning component.

It follows an iterative process in which the agent executes the planned actions, observes the tool outputs, and collects the retrieved evidence.

After completing the planned steps, the agent generates a final response based on the collected results.

The process stops when all planned actions are completed or the maximum number of iterations is reached.


In [14]:

def run_react_agent(user_request, execution_plan):

    observations = []
    execution_log = []
    used_tools = []

    # 1. Validate the execution plan.
    if not isinstance(execution_plan, list) or not execution_plan:
        return {
            "status": "error",
            "error": "The execution plan is empty or invalid.",
            "iterations": 0,
            "used_tools": [],
            "observations": [],
            "execution_log": [],
            "final_response": None
        }

    planned_tools = []

    for step in execution_plan:

        if not isinstance(step, dict):
            return {
                "status": "error",
                "error": "Invalid execution plan step."
            }

        tool_name = step.get("tool_name")

        if tool_name not in TOOL_REGISTRY:
            return {
                "status": "error",
                "error": f"Unknown tool in plan: {tool_name}"
            }

        if tool_name in planned_tools:
            return {
                "status": "error",
                "error": f"Repeated tool in plan: {tool_name}"
            }

        planned_tools.append(tool_name)

    # 2. Execute tools according to the plan.
    for iteration, step in enumerate(execution_plan, start=1):

        tool_name = step["tool_name"]

        tool_result = execute_tool(tool_name)
        success = tool_result.get("status") == "success"

        used_tools.append(tool_name)

        execution_log.append({
            "iteration": iteration,
            "tool_name": tool_name,
            "purpose": step.get("purpose", ""),
            "execution_status": tool_result.get("status"),
            "execution_time": tool_result.get("execution_time")
        })

        if not success:
            return {
                "status": "error",
                "error": f"Tool execution failed: {tool_name}",
                "iterations": iteration,
                "used_tools": used_tools,
                "observations": observations,
                "execution_log": execution_log,
                "final_response": None
            }

        observations.append({
            "iteration": iteration,
            "tool_name": tool_name,
            "status": "success",
            "result": tool_result.get("result")
        })

    # 3. Collect evidence from tool observations.
    evidence = {
        item["tool_name"]: item["result"]
        for item in observations
    }

    # 4. Retrieve performance and drift results.
    performance = evidence.get("get_performance_baseline")
    drift = evidence.get("get_drift_analysis")

    # Validate required evidence.
    if not isinstance(performance, dict) or not isinstance(drift, dict):
        return {
            "status": "incomplete",
            "error": "Performance or drift evidence is missing.",
            "iterations": len(execution_log),
            "used_tools": used_tools,
            "observations": observations,
            "execution_log": execution_log,
            "final_response": None
        }

    # 5. Validate performance metrics.
    metrics = performance.get("metrics")

    required_metrics = [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-Score",
        "ROC-AUC",
        "PR-AUC"
    ]

    if not isinstance(metrics, dict):
        return {
            "status": "incomplete",
            "error": "Performance metrics are missing or invalid.",
            "iterations": len(execution_log),
            "used_tools": used_tools,
            "observations": observations,
            "execution_log": execution_log,
            "final_response": None
        }

    missing_metrics = [
        metric for metric in required_metrics
        if metric not in metrics
    ]

    if missing_metrics:
        return {
            "status": "incomplete",
            "error": f"Missing metrics: {missing_metrics}",
            "iterations": len(execution_log),
            "used_tools": used_tools,
            "observations": observations,
            "execution_log": execution_log,
            "final_response": None
        }

    # Validate metric values.
    for metric in required_metrics:
        value = metrics[metric]

        if (
            not isinstance(value, (int, float))
            or isinstance(value, bool)
            or not 0 <= value <= 1
        ):
            return {
                "status": "incomplete",
                "error": f"Invalid metric value: {metric}",
                "iterations": len(execution_log),
                "used_tools": used_tools,
                "observations": observations,
                "execution_log": execution_log,
                "final_response": None
            }

    # 6. Validate drift results.
    number_of_features = drift.get("number_of_features")
    drifted_features = drift.get("drifted_features")

    if (
        not isinstance(number_of_features, int)
        or isinstance(number_of_features, bool)
        or number_of_features < 1
        or not isinstance(drifted_features, list)
    ):
        return {
            "status": "incomplete",
            "error": "Drift analysis results are invalid.",
            "iterations": len(execution_log),
            "used_tools": used_tools,
            "observations": observations,
            "execution_log": execution_log,
            "final_response": None
        }

    if len(drifted_features) > number_of_features:
        return {
            "status": "incomplete",
            "error": "Drifted feature count exceeds analyzed features.",
            "iterations": len(execution_log),
            "used_tools": used_tools,
            "observations": observations,
            "execution_log": execution_log,
            "final_response": None
        }

    # 7. Generate performance report using Python.
    metric_text = "\n".join(
        f"- **{name}:** {metrics[name]:.4f}"
        for name in required_metrics
    )

    # 8. Generate drift report using Python.
    if drifted_features:

        drift_text = (
            "Statistically significant drift was detected in: "
            + ", ".join(drifted_features)
            + "."
        )

    else:

        drift_text = (
            f"No statistically significant drift was detected "
            f"across the {number_of_features} analyzed features."
        )

    # 9. Build the final evidence-based response.
    final_response = f"""
### Churn Prediction Model Assessment

**Model:** {performance.get("model", "Not specified")}

**Evaluation Dataset:** {performance.get("evaluation_dataset", "Not specified")}

**Classification Threshold:** {performance.get("threshold", "Not specified")}

#### Baseline Performance Metrics

{metric_text}

#### Drift Analysis

**Analysis Type:** {drift.get("analysis_type", "Not specified")}

**Number of Analyzed Features:** {number_of_features}

**Number of Drifted Features:** {len(drifted_features)}

{drift_text}

#### Interpretation

The XGBoost model achieved the reported performance
metrics on the held-out test dataset.

The drift analysis did not identify statistically
significant distribution changes in the analyzed
features, based on the applied statistical tests
and significance threshold.

However, the absence of detected data drift does
not prove that model performance will remain stable.

#### Limitations

The drift analysis compares reference and comparison
datasets. It does not represent live production monitoring.

Further monitoring using new production data and
actual customer outcomes is required to assess
long-term model performance.
"""

    # 10. Return the execution results.
    return {
        "status": "completed",
        "iterations": len(execution_log),
        "used_tools": used_tools,
        "observations": observations,
        "execution_log": execution_log,
        "final_response": final_response
    }


print("Planning and ReAct integration completed.")

Planning and ReAct integration completed.


### ReAct Controller Testing

This step tests the integration between the Reasoning and Planning component and the ReAct Controller using a request that requires multiple tools.

The agent first generates an execution plan, then the ReAct Controller executes the planned tools, observes their outputs, and collects the results.

The test verifies whether the generated plan is followed correctly and whether the agent can combine the retrieved information into one evidence-based response.

In [15]:

test_request = """
Assess the churn prediction model.

Check whether statistically significant data drift was detected
and retrieve the model's baseline performance metrics.

Combine both results into one evidence-based response.
"""

# Step 1: Generate the execution plan.
planning_result = reason_and_plan(test_request)

if planning_result["status"] != "success":
    raise RuntimeError(
        f"Planning failed: {planning_result}"
    )

execution_plan = planning_result["plan"]

print("Generated Execution Plan:")
for step in execution_plan:
    print(step)

# Step 2: Execute the generated plan.
react_result = run_react_agent(
    user_request=test_request,
    execution_plan=execution_plan
)

print("\nStatus:", react_result["status"])
print("Iterations:", react_result["iterations"])

print("\nTools Used:")
for tool in react_result["used_tools"]:
    print("-", tool)

print("\nExecution Log:")
for step in react_result["execution_log"]:
    print(step)

# Step 3: Verify plan execution.
planned_tools = [
    step["tool_name"] for step in execution_plan
]

executed_tools = react_result["used_tools"]

print("\nPlan Execution Check:")
print("Plan followed exactly:", planned_tools == executed_tools)

# Step 4: Display the final response.
print("\nFinal Response:")
print(react_result.get("final_response"))


Generated Execution Plan:
{'step': 1, 'tool_name': 'get_model_information', 'purpose': 'Retrieve model information, configurations, and evaluation results.'}
{'step': 2, 'tool_name': 'get_performance_baseline', 'purpose': 'Retrieve baseline performance metrics of the churn prediction model.'}
{'step': 3, 'tool_name': 'get_drift_analysis', 'purpose': 'Analyze statistical data drift between reference and comparison datasets.'}

Status: completed
Iterations: 3

Tools Used:
- get_model_information
- get_performance_baseline
- get_drift_analysis

Execution Log:
{'iteration': 1, 'tool_name': 'get_model_information', 'purpose': 'Retrieve model information, configurations, and evaluation results.', 'execution_status': 'success', 'execution_time': 0.0021}
{'iteration': 2, 'tool_name': 'get_performance_baseline', 'purpose': 'Retrieve baseline performance metrics of the churn prediction model.', 'execution_status': 'success', 'execution_time': 0.001}
{'iteration': 3, 'tool_name': 'get_drift_analy

## Memory Management

Memory allows the ModelGuard Agent to retain relevant information during task execution and across different sessions.

The memory system consists of two components:

- **Short-Term Memory:** Stores the current user request, execution plan, tool results, and observations during an active session.
- **Persistent Memory:** Stores selected and verified system information that can be reused in future sessions.

The agent retrieves relevant memory when needed to improve task continuity and reduce repeated operations.

Only relevant and verified information should be stored. Sensitive customer data and unsupported LLM-generated assumptions must not be saved.

In [16]:
import json
from pathlib import Path
from datetime import datetime
import uuid

# Create memory directory
MEMORY_DIR = PROJECT_ROOT / "memory"
MEMORY_DIR.mkdir(parents=True, exist_ok=True)

# Persistent memory file
PERSISTENT_MEMORY_FILE = MEMORY_DIR / "persistent_memory.json"

# Initialize persistent memory if it does not exist
if not PERSISTENT_MEMORY_FILE.exists():
    initial_memory = {
        "system_facts": [],
        "verified_observations": []
    }

    with open(PERSISTENT_MEMORY_FILE, "w", encoding="utf-8") as file:
        json.dump(initial_memory, file, indent=4)

# Initialize short-term memory for the current session
SESSION_ID = str(uuid.uuid4())

short_term_memory = {
    "session_id": SESSION_ID,
    "created_at": datetime.now().isoformat(),
    "user_requests": [],
    "execution_plans": [],
    "tool_observations": []
}

print("Memory system initialized successfully.")
print(f"Memory directory: {MEMORY_DIR}")
print(f"Session ID: {SESSION_ID}")
print(f"Persistent memory file: {PERSISTENT_MEMORY_FILE}")

Memory system initialized successfully.
Memory directory: C:\Users\User\ModelGuard\memory
Session ID: 19db9feb-88cd-4c39-8e76-e266eb9fe108
Persistent memory file: C:\Users\User\ModelGuard\memory\persistent_memory.json


### Memory Functions

In this step, functions are developed to manage the agent's memory operations.

The functions support loading and saving persistent memory, adding information to short-term memory, storing verified information for future sessions, and retrieving relevant entries using keyword matching.

These functions provide the basic memory operations required by the ModelGuard Agent.

In [17]:
def load_persistent_memory():
    """Load verified information from persistent memory."""
    with open(PERSISTENT_MEMORY_FILE, "r", encoding="utf-8") as file:
        return json.load(file)


def save_persistent_memory(memory_data):
    """Save persistent memory to JSON."""
    with open(PERSISTENT_MEMORY_FILE, "w", encoding="utf-8") as file:
        json.dump(memory_data, file, indent=4, ensure_ascii=False)


def add_short_term_memory(category, content):
    """Store information during the current session."""
    if category not in short_term_memory:
        raise ValueError(f"Unknown memory category: {category}")

    short_term_memory[category].append({
        "timestamp": datetime.now().isoformat(),
        "content": content
    })


def add_persistent_memory(category, content, source):
    """Store verified information without creating duplicates."""

    allowed_categories = ["system_facts", "verified_observations"]

    if category not in allowed_categories:
        raise ValueError("Invalid persistent memory category.")

    if not content or not source:
        raise ValueError("Content and source are required.")

    memory_data = load_persistent_memory()

    # Check for duplicate information
    for entry in memory_data[category]:
        if entry["content"] == content and entry["source"] == source:
            print("Information already exists in persistent memory.")
            return

    memory_data[category].append({
        "timestamp": datetime.now().isoformat(),
        "content": content,
        "source": source
    })

    save_persistent_memory(memory_data)


def retrieve_memory(query):
    """Retrieve relevant entries using simple keyword matching."""
    memory_data = load_persistent_memory()
    query_words = set(query.lower().split())

    results = []

    for category, entries in memory_data.items():
        for entry in entries:
            searchable_text = entry["content"].lower()
            if any(word in searchable_text for word in query_words):
                results.append({
                    "category": category,
                    **entry
                })

    return results

print("Information saved successfully.")
print("Memory functions defined successfully.")

Information saved successfully.
Memory functions defined successfully.


### Persistent Memory Deduplication

This step removes duplicate entries from persistent memory while preserving the original information and its source.

The process ensures that each unique piece of information is stored only once within its category.

### Remove Duplicate Entries

In [18]:
memory_data = load_persistent_memory()

for category, entries in memory_data.items():
    unique_entries = []
    seen = set()

    for entry in entries:
        key = (entry["content"], entry["source"])

        if key not in seen:
            unique_entries.append(entry)
            seen.add(key)

    memory_data[category] = unique_entries

save_persistent_memory(memory_data)

print("Memory deduplication completed.")

for category, entries in memory_data.items():
    print(f"{category}: {len(entries)} unique entries")

Memory deduplication completed.
system_facts: 1 unique entries
verified_observations: 0 unique entries


### Testing Duplicate Prevention

In [19]:
test_fact = "The churn prediction model is an XGBoost classifier."
test_source = "artifacts/model_results.json"

# Attempt to add the same information twice
add_persistent_memory("system_facts", test_fact, test_source)
add_persistent_memory("system_facts", test_fact, test_source)

# Verify the number of unique entries
memory_data = load_persistent_memory()

matching_entries = [
    entry for entry in memory_data["system_facts"]
    if entry["content"] == test_fact and entry["source"] == test_source
]

print(f"\nMatching entries found: {len(matching_entries)}")
print("Duplicate prevention test:", "PASSED" if len(matching_entries) == 1 else "FAILED")

Information already exists in persistent memory.
Information already exists in persistent memory.

Matching entries found: 1
Duplicate prevention test: PASSED


### Memory Operations Testing

In this step, the memory functions are tested to verify that information can be stored and retrieved correctly.

The test covers both short-term and persistent memory. It also checks whether the retrieval function can identify relevant information using keyword matching.

In [20]:
# Test short-term memory
add_short_term_memory(
    "user_requests",
    "Check the churn model performance."
)

add_short_term_memory(
    "tool_observations",
    "The XGBoost model achieved an F1-score of 0.6270."
)

# Test persistent memory using verified project information
add_persistent_memory(
    category="system_facts",
    content="The churn prediction model is an XGBoost classifier.",
    source="artifacts/model_results.json"
)

# Retrieve stored information
retrieved_results = retrieve_memory("XGBoost churn model")

print("Short-Term Memory:")
print(json.dumps(short_term_memory, indent=4, ensure_ascii=False))

print("\nRetrieved Persistent Memory:")
print(json.dumps(retrieved_results, indent=4, ensure_ascii=False))

Information already exists in persistent memory.
Short-Term Memory:
{
    "session_id": "19db9feb-88cd-4c39-8e76-e266eb9fe108",
    "created_at": "2026-11-25T18:11:17.523372",
    "user_requests": [
        {
            "timestamp": "2026-11-25T18:11:17.739614",
            "content": "Check the churn model performance."
        }
    ],
    "execution_plans": [],
    "tool_observations": [
        {
            "timestamp": "2026-11-25T18:11:17.739614",
            "content": "The XGBoost model achieved an F1-score of 0.6270."
        }
    ]
}

Retrieved Persistent Memory:
[
    {
        "category": "system_facts",
        "timestamp": "2026-11-22T18:21:34.583226",
        "content": "The churn prediction model is an XGBoost classifier.",
        "source": "artifacts/model_results.json"
    }
]


### Persistent Memory Verification

This step verifies that stored information can be loaded again from the persistent memory file.

The test checks whether previously saved system information remains available independently of the current memory object.

In [21]:
# Reload persistent memory directly from the JSON file
reloaded_memory = load_persistent_memory()

# Verify the stored information
assert len(reloaded_memory["system_facts"]) > 0

print("Persistent memory verification completed.")

for item in reloaded_memory["system_facts"]:
    print(f"- Information: {item['content']}")
    print(f"  Source: {item['source']}")

Persistent memory verification completed.
- Information: The churn prediction model is an XGBoost classifier.
  Source: artifacts/model_results.json


### Memory Integration with the Agent

In this step, the memory system is connected to the ModelGuard Agent workflow.

The agent can retrieve relevant information from persistent memory and use it as additional context when processing user requests.

Short-term memory is also updated during task execution to maintain the context of the current session.

This integration helps the agent maintain continuity and reuse previously verified system information.

In [22]:
# Restore memory environment after kernel restart

import json
import uuid
from pathlib import Path
from datetime import datetime

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

MEMORY_DIR = PROJECT_ROOT / "memory"
PERSISTENT_MEMORY_FILE = MEMORY_DIR / "persistent_memory.json"

MEMORY_DIR.mkdir(parents=True, exist_ok=True)

# Initialize a new short-term memory session
SESSION_ID = str(uuid.uuid4())

short_term_memory = {
    "session_id": SESSION_ID,
    "created_at": datetime.now().isoformat(),
    "user_requests": [],
    "execution_plans": [],
    "tool_observations": []
}


def load_persistent_memory():
    with open(PERSISTENT_MEMORY_FILE, "r", encoding="utf-8") as file:
        return json.load(file)


def add_short_term_memory(category, content):
    if category not in short_term_memory:
        raise ValueError(f"Unknown memory category: {category}")

    short_term_memory[category].append({
        "timestamp": datetime.now().isoformat(),
        "content": content
    })


def retrieve_memory(query):
    memory_data = load_persistent_memory()
    query_words = set(query.lower().split())

    results = []

    for category, entries in memory_data.items():
        for entry in entries:
            searchable_text = entry["content"].lower()

            if any(word in searchable_text for word in query_words):
                results.append({
                    "category": category,
                    **entry
                })

    return results


print("Agent memory components restored successfully.")
print(f"Session ID: {SESSION_ID}")
print(f"Persistent memory entries: {sum(len(v) for v in load_persistent_memory().values())}")

Agent memory components restored successfully.
Session ID: b61a1ad4-353a-48bf-8256-599d54527398
Persistent memory entries: 1


### Testing Memory-Aware Planning

This test evaluates whether the agent can retrieve previously stored information and use it as additional context when generating an execution plan.

The test checks whether relevant memory is retrieved, the user request is processed correctly, and an appropriate execution plan is generated.

The results are used to verify the integration between the memory system and the agent's planning component.

In [23]:
def reason_and_plan_with_memory(user_request):
    """Generate an execution plan using relevant persistent memory."""

    # Retrieve relevant information
    memory_context = retrieve_memory(user_request)

    # Record the original request in short-term memory
    add_short_term_memory("user_requests", user_request)

    # Prepare additional context
    if memory_context:
        context_text = "\n".join([
            f"- {item['content']} (Source: {item['source']})"
            for item in memory_context
        ])

        augmented_request = (
            f"User request: {user_request}\n\n"
            f"Verified memory context:\n{context_text}\n\n"
            "Use this context only when relevant. "
            "Do not treat it as a replacement for current tool results."
        )
    else:
        augmented_request = user_request

    # Generate the execution plan
    reasoning_result = reason_and_plan(augmented_request)

    # Store the generated plan
    add_short_term_memory(
        "execution_plans",
        reasoning_result
    )

    return {
        "user_request": user_request,
        "retrieved_memory": memory_context,
        "planning_result": reasoning_result
    }


print("Memory-aware planning function defined successfully.")


Memory-aware planning function defined successfully.


### Testing Memory-Aware Planning

This test evaluates whether the agent can retrieve previously stored information and use it as additional context when generating an execution plan.

The test checks memory retrieval, request processing, and plan generation.

In [24]:
# Define a test request
test_request = "What model is used for churn prediction?"

# Execute memory-aware planning
memory_planning_result = reason_and_plan_with_memory(test_request)

# Display the test results
print("User Request:")
print(memory_planning_result["user_request"])

print("\nRetrieved Memory:")
print(json.dumps(
    memory_planning_result["retrieved_memory"],
    indent=4,
    ensure_ascii=False
))

print("\nGenerated Execution Plan:")
print(json.dumps(
    memory_planning_result["planning_result"],
    indent=4,
    ensure_ascii=False
))

User Request:
What model is used for churn prediction?

Retrieved Memory:
[
    {
        "category": "system_facts",
        "timestamp": "2026-11-22T18:21:34.583226",
        "content": "The churn prediction model is an XGBoost classifier.",
        "source": "artifacts/model_results.json"
    }
]

Generated Execution Plan:
{
    "status": "success",
    "reasoning": {
        "objective": "The user is asking for information about the churn prediction model used.",
        "required_information": [
            "The model name and type (XGBoost classifier) for the churn prediction model."
        ]
    },
    "plan": [
        {
            "step": 1,
            "tool_name": "get_model_information",
            "purpose": "Retrieve information about the churn prediction model."
        }
    ]
}


## Structured Output and Validation

Structured Output ensures that the ModelGuard Agent produces information in a predefined and consistent format.

The agent uses JSON to represent structured responses, while Pydantic is used to validate the output structure and data types.

This approach helps detect missing fields, invalid values, and incorrect data formats before the results are used by other system components.

### Pydantic Setup

In this step, Pydantic is imported to define and validate structured data models.

These models will be used to check the format and required fields of the agent's outputs.

In [25]:
import pydantic
from pydantic import BaseModel, Field, ValidationError
from typing import List

print("Pydantic version:", pydantic.__version__)
print("Structured output environment initialized.")

Pydantic version: 2.14.4
Structured output environment initialized.


### Defining Output Schemas

In this step, Pydantic models are defined to specify the expected structure of the agent's planning output.

The schema includes the execution status, reasoning objective, required information, and execution plan.

Each execution step must contain a step number, a valid tool name, and a description of its purpose. These definitions provide a consistent structure for validating agent outputs.

In [26]:
from typing import Literal
from pydantic import BaseModel, Field, model_validator


class PlanStep(BaseModel):
    step: int = Field(ge=1)
    tool_name: str = Field(min_length=1)
    purpose: str = Field(min_length=1)


class ReasoningDetails(BaseModel):
    objective: str = Field(min_length=1)
    required_information: List[str]


class PlanningOutput(BaseModel):
    status: Literal["success"]
    reasoning: ReasoningDetails
    plan: List[PlanStep] = Field(max_length=5)

    @model_validator(mode="after")
    def validate_plan(self):
        step_numbers = [item.step for item in self.plan]
        tool_names = [item.tool_name for item in self.plan]

        if step_numbers != list(range(1, len(self.plan) + 1)):
            raise ValueError("Plan steps must be sequential.")

        if len(tool_names) != len(set(tool_names)):
            raise ValueError("Duplicate tools are not allowed.")

        return self


print("Pydantic output schemas defined successfully.")

Pydantic output schemas defined successfully.


### Validating Agent Planning Output

In this step, the planning output generated by the agent is validated against the predefined Pydantic schema.

The validation checks required fields, data types, sequential step numbers, and duplicate tool usage.

This ensures that only structurally valid planning outputs can proceed to the execution stage.

In [27]:
# Generate a planning result
test_request = "Check the churn model performance and identify the model used."

raw_planning_output = reason_and_plan(test_request)

# Validate the generated output
try:
    validated_plan = PlanningOutput.model_validate(raw_planning_output)

    print("Planning output validation: PASSED")
    print("\nValidated JSON output:")

    print(json.dumps(
        validated_plan.model_dump(),
        indent=4,
        ensure_ascii=False
    ))

except ValidationError as error:
    print("Planning output validation: FAILED")
    print(error)

Planning output validation: PASSED

Validated JSON output:
{
    "status": "success",
    "reasoning": {
        "objective": "The user wants to check the churn model performance and identify the model used.",
        "required_information": [
            "The model's performance metrics and the model's configuration details."
        ]
    },
    "plan": [
        {
            "step": 1,
            "tool_name": "get_model_information",
            "purpose": "Retrieve the model's configuration and evaluation results."
        },
        {
            "step": 2,
            "tool_name": "get_performance_baseline",
            "purpose": "Retrieve the baseline performance metrics of the churn prediction model."
        }
    ]
}


### Tool Registry Validation

In this step, the validated planning output is checked against the available tool registry.

Each tool name must exist in the TOOL_REGISTRY before execution. This prevents the agent from selecting undefined tools and reduces execution errors.

The validation also ensures that the execution plan contains only tools supported by the ModelGuard system.

In [28]:
def validate_planning_tools(validated_plan):
    """Check whether all planned tools exist in the tool registry."""

    available_tools = set(TOOL_REGISTRY.keys())

    planned_tools = [
        step.tool_name
        for step in validated_plan.plan
    ]

    invalid_tools = [
        tool for tool in planned_tools
        if tool not in available_tools
    ]

    if invalid_tools:
        raise ValueError(f"Unknown tools detected: {invalid_tools}")

    return True


try:
    validate_planning_tools(validated_plan)

    print("Tool registry validation: PASSED")
    print(f"Validated tools: {len(validated_plan.plan)}")

    for step in validated_plan.plan:
        print(f"- {step.tool_name}: Available")

except ValueError as error:
    print("Tool registry validation: FAILED")
    print(error)

Tool registry validation: PASSED
Validated tools: 2
- get_model_information: Available
- get_performance_baseline: Available


### Invalid Output Testing

In this step, invalid planning outputs are intentionally introduced to test the validation mechanism.

The tests include missing required fields, invalid step sequences, and duplicate tool names.

The objective is to verify that Pydantic rejects outputs that do not follow the predefined schema.

In [29]:
invalid_test_cases = {
    "Missing Required Field": {
        "status": "success",
        "reasoning": {
            "objective": "Test invalid output",
            "required_information": ["Model information"]
        },
        "plan": [
            {
                "step": 1,
                "tool_name": "get_model_information"
                # Missing purpose
            }
        ]
    },

    "Invalid Step Sequence": {
        "status": "success",
        "reasoning": {
            "objective": "Test invalid step sequence",
            "required_information": ["Model information"]
        },
        "plan": [
            {
                "step": 2,
                "tool_name": "get_model_information",
                "purpose": "Retrieve model information"
            }
        ]
    },

    "Duplicate Tools": {
        "status": "success",
        "reasoning": {
            "objective": "Test duplicate tools",
            "required_information": ["Model information"]
        },
        "plan": [
            {
                "step": 1,
                "tool_name": "get_model_information",
                "purpose": "Retrieve model information"
            },
            {
                "step": 2,
                "tool_name": "get_model_information",
                "purpose": "Retrieve model information again"
            }
        ]
    }
}

validation_results = {}

for test_name, invalid_data in invalid_test_cases.items():
    try:
        PlanningOutput.model_validate(invalid_data)
        validation_results[test_name] = "NOT REJECTED"
    except ValidationError:
        validation_results[test_name] = "REJECTED"

print("Invalid Output Validation Results:")

for test_name, result in validation_results.items():
    print(f"{test_name}: {result}")

passed_tests = sum(
    result == "REJECTED"
    for result in validation_results.values()
)

print(f"\nTests Passed: {passed_tests}/{len(invalid_test_cases)}")

Invalid Output Validation Results:
Missing Required Field: REJECTED
Invalid Step Sequence: REJECTED
Duplicate Tools: REJECTED

Tests Passed: 3/3


### Handling Invalid Outputs

In this step, a validation wrapper is implemented to handle invalid planning outputs.

The function validates the agent's output against the Pydantic schema and checks whether all selected tools exist in the tool registry.

If validation fails, the function returns a structured error instead of allowing an invalid plan to proceed to execution.

In [30]:
def validate_agent_output(raw_output):
    """Validate planning output and check tool availability."""

    try:
        # Validate output structure
        validated_output = PlanningOutput.model_validate(raw_output)

        # Validate tool names
        validate_planning_tools(validated_output)

        return {
            "status": "valid",
            "data": validated_output.model_dump(),
            "error": None
        }

    except ValidationError as error:
        return {
            "status": "invalid",
            "data": None,
            "error": error.errors()
        }

    except ValueError as error:
        return {
            "status": "invalid",
            "data": None,
            "error": str(error)
        }


# Test the handler using an invalid output
test_invalid_output = invalid_test_cases["Missing Required Field"]

handler_result = validate_agent_output(test_invalid_output)

print("Validation Handler Result:")
print(json.dumps(handler_result, indent=4, ensure_ascii=False))

Validation Handler Result:
{
    "status": "invalid",
    "data": null,
    "error": [
        {
            "type": "missing",
            "loc": [
                "plan",
                0,
                "purpose"
            ],
            "msg": "Field required",
            "input": {
                "step": 1,
                "tool_name": "get_model_information"
            },
            "url": "https://errors.pydantic.dev/2.14/v/missing"
        }
    ]
}


### Valid Output Acceptance Test

This test verifies that the validation handler accepts correctly structured planning outputs.

It checks both schema compliance and tool availability to ensure that valid plans can proceed to the execution stage.

In [31]:
 # Generate a valid planning output
valid_test_output = reason_and_plan(
    "Check the churn model performance."
)

# Validate the generated output
valid_handler_result = validate_agent_output(valid_test_output)

print("Valid Output Test:")
print(json.dumps(
    valid_handler_result,
    indent=4,
    ensure_ascii=False
))

print("\nTest Result:",
      "PASSED" if valid_handler_result["status"] == "valid" else "FAILED")

Valid Output Test:
{
    "status": "valid",
    "data": {
        "status": "success",
        "reasoning": {
            "objective": "Check the churn model performance",
            "required_information": [
                "Retrieve model information, configurations, and evaluation results",
                "Retrieve baseline performance metrics of the churn prediction model"
            ]
        },
        "plan": [
            {
                "step": 1,
                "tool_name": "get_model_information",
                "purpose": "Retrieve model information, configurations, and evaluation results"
            },
            {
                "step": 2,
                "tool_name": "get_performance_baseline",
                "purpose": "Retrieve baseline performance metrics of the churn prediction model"
            }
        ]
    },
    "error": null
}

Test Result: PASSED


### Structured Output Integration

In this step, structured output validation is integrated into the agent's planning workflow.

The generated planning output is validated automatically before being returned. Invalid outputs are rejected, while valid outputs are converted into a standardized format.

This integration ensures consistent output validation before tool execution.

In [32]:
def generate_validated_plan(user_request):
    """Generate and validate the agent's execution plan."""

    raw_output = reason_and_plan(user_request)

    validation_result = validate_agent_output(raw_output)

    if validation_result["status"] != "valid":
        return validation_result

    return validation_result


# Integration test
integration_result = generate_validated_plan(
    "Check the churn model performance and identify potential risks."
)

print("Structured Output Integration:")
print(json.dumps(
    integration_result,
    indent=4,
    ensure_ascii=False
))

Structured Output Integration:
{
    "status": "valid",
    "data": {
        "status": "success",
        "reasoning": {
            "objective": "Check the churn model performance and identify potential risks.",
            "required_information": [
                "Retrieve model information, configurations, and evaluation results.",
                "Retrieve baseline performance metrics of the churn prediction model.",
                "Analyze statistical data drift between reference and comparison datasets."
            ]
        },
        "plan": [
            {
                "step": 1,
                "tool_name": "get_model_information",
                "purpose": "Retrieve model information, configurations, and evaluation results."
            },
            {
                "step": 2,
                "tool_name": "get_performance_baseline",
                "purpose": "Retrieve baseline performance metrics of the churn prediction model."
            },
            {
      

### Final Response Schema

In this step, a Pydantic schema is defined to standardize the agent's final responses.

The schema separates the main answer, key findings, supporting evidence, and limitations.

This ensures that final responses follow a consistent structure and can be validated before being presented to the user.

In [33]:
class FinalResponse(BaseModel):
    status: Literal["success", "error"]
    answer: str = Field(min_length=1)
    key_findings: List[str]
    evidence: List[str]
    limitations: List[str]

### Inspecting ReAct Output

In [34]:
# Generate and validate an execution plan
test_request = "Check the churn model performance."

planning_result = reason_and_plan(test_request)
validated_plan = PlanningOutput.model_validate(planning_result)
validate_planning_tools(validated_plan)

# Execute the plan using the ReAct controller
react_result = run_react_agent(
    test_request,
    validated_plan.model_dump()["plan"]
)

# Inspect the actual output structure
print("ReAct execution status:", react_result.get("status"))
print("\nAvailable output fields:", list(react_result.keys()))

print("\nReAct output:")
print(json.dumps(react_result, indent=4, ensure_ascii=False, default=str))

ReAct execution status: incomplete

Available output fields: ['status', 'error', 'iterations', 'used_tools', 'observations', 'execution_log', 'final_response']

ReAct output:
{
    "status": "incomplete",
    "error": "Performance or drift evidence is missing.",
    "iterations": 2,
    "used_tools": [
        "get_model_information",
        "get_performance_baseline"
    ],
    "observations": [
        {
            "iteration": 1,
            "tool_name": "get_model_information",
            "status": "success",
            "result": {
                "model_results": {
                    "model_role": "Final Model",
                    "cross_validation": {
                        "Model": "XGBoost",
                        "precision": 0.5326540508909394,
                        "precision_std": 0.007268877099787079,
                        "recall": 0.7759197324414716,
                        "recall_std": 0.020832657528290912,
                        "f1": 0.6315611506372274,


### Final Response Generation and Validation

In [35]:
class FinalResponse(BaseModel):
    status: Literal["success", "partial", "error"]
    answer: str = Field(min_length=1)
    key_findings: List[str]
    evidence: List[str]
    limitations: List[str]

In [36]:
# Extract available evidence from ReAct observations
observations = react_result["observations"]

performance = next(
    (
        item["result"]
        for item in observations
        if item["tool_name"] == "get_performance_baseline"
        and item["status"] == "success"
    ),
    None
)

model_info = next(
    (
        item["result"]
        for item in observations
        if item["tool_name"] == "get_model_information"
        and item["status"] == "success"
    ),
    None
)

if performance and model_info:
    metrics = performance["metrics"]

    response_data = {
        "status": "partial" if react_result["status"] != "completed" else "success",
        "answer": (
            f"The churn prediction model uses {performance['model']}. "
            f"Its F1-score is {metrics['F1-Score']:.4f} "
            f"and its ROC-AUC is {metrics['ROC-AUC']:.4f}."
        ),
        "key_findings": [
            f"Accuracy: {metrics['Accuracy']:.4f}",
            f"Precision: {metrics['Precision']:.4f}",
            f"Recall: {metrics['Recall']:.4f}",
            f"F1-score: {metrics['F1-Score']:.4f}",
            f"ROC-AUC: {metrics['ROC-AUC']:.4f}",
            f"PR-AUC: {metrics['PR-AUC']:.4f}"
        ],
        "evidence": [
            "get_model_information",
            "get_performance_baseline"
        ],
        "limitations": [
            "Results are based on the held-out test dataset.",
            f"ReAct execution status: {react_result['status']}.",
            f"Execution issue: {react_result['error']}"
        ]
    }

    validated_response = FinalResponse.model_validate(response_data)

    print(json.dumps(
        validated_response.model_dump(),
        indent=4,
        ensure_ascii=False
    ))

else:
    print("Final response cannot be generated: required evidence is missing.")

{
    "status": "partial",
    "answer": "The churn prediction model uses XGBoost. Its F1-score is 0.6270 and its ROC-AUC is 0.8463.",
    "key_findings": [
        "Accuracy: 0.7644",
        "Precision: 0.5407",
        "Recall: 0.7460",
        "F1-score: 0.6270",
        "ROC-AUC: 0.8463",
        "PR-AUC: 0.6622"
    ],
    "evidence": [
        "get_model_information",
        "get_performance_baseline"
    ],
    "limitations": [
        "Results are based on the held-out test dataset.",
        "ReAct execution status: incomplete.",
        "Execution issue: Performance or drift evidence is missing."
    ]
}


## Reflection and Self-Correction

Reflection allows the ModelGuard Agent to review its generated responses before presenting them to the user.

The reflection component checks whether the response is supported by tool outputs, identifies missing evidence and contradictions, and determines whether the response requires correction.

This process improves response reliability and reduces unsupported claims.

### Reflection Schema

A structured reflection model is defined to record the review result, identified issues, and required corrective actions.

In [37]:
class ReflectionOutput(BaseModel):
    status: Literal["approved", "needs_correction", "rejected"]
    evidence_supported: bool
    identified_issues: List[str]
    correction_required: bool

### Implement Evidence-Based Reflection

The reflection function compares the final response with the available tool observations.

It checks whether numerical findings are supported by the retrieved evidence and whether the response accurately represents the execution status.

In [38]:
def reflect_on_response(response_data, react_result):
    """Review final response against actual tool observations."""

    issues = []
    supported_claims = 0
    checked_claims = 0

    observations = [
        item for item in react_result.get("observations", [])
        if item.get("status") == "success"
    ]

    performance = next(
        (
            item["result"]
            for item in observations
            if item["tool_name"] == "get_performance_baseline"
        ),
        None
    )

    if performance:
        expected_metrics = performance.get("metrics", {})

        metric_map = {
            "Accuracy": "Accuracy",
            "Precision": "Precision",
            "Recall": "Recall",
            "F1-score": "F1-Score",
            "ROC-AUC": "ROC-AUC",
            "PR-AUC": "PR-AUC"
        }

        for label, key in metric_map.items():
            matching_claims = [
                item for item in response_data.key_findings
                if item.startswith(label + ":")
            ]

            if matching_claims:
                checked_claims += 1

                try:
                    reported = float(
                        matching_claims[0].split(":", 1)[1].strip()
                    )
                    actual = expected_metrics[key]

                    if abs(reported - actual) <= 0.0001:
                        supported_claims += 1
                    else:
                        issues.append(f"{label} does not match evidence.")

                except (ValueError, KeyError):
                    issues.append(f"{label} has an invalid value.")

    else:
        issues.append("Performance evidence is missing.")

    # Check whether incomplete execution is disclosed
    if react_result.get("status") != "completed":
        disclosure = " ".join(
            [response_data.answer] + response_data.limitations
        ).lower()

        if "incomplete" not in disclosure:
            issues.append(
                "Incomplete execution is not clearly disclosed."
            )

    evidence_supported = (
        checked_claims > 0 and supported_claims == checked_claims
    )

    correction_required = bool(issues)

    return ReflectionOutput(
        status="needs_correction" if correction_required else "approved",
        evidence_supported=evidence_supported,
        identified_issues=issues,
        correction_required=correction_required
    )


print("Reflection validation updated successfully.")

Reflection validation updated successfully.


### Reflection Testing

In [39]:
reflection_result = reflect_on_response(
    validated_response,
    react_result
)

print(json.dumps(
    reflection_result.model_dump(),
    indent=4,
    ensure_ascii=False
))

{
    "status": "approved",
    "evidence_supported": true,
    "identified_issues": [],
    "correction_required": false
}


### Incorrect Claim Detection

In [40]:
# Create a copy of the valid response
incorrect_response_data = validated_response.model_copy(deep=True)

# Intentionally introduce an incorrect metric
incorrect_response_data.key_findings[0] = "Accuracy: 0.9500"

# Run reflection
incorrect_reflection = reflect_on_response(
    incorrect_response_data,
    react_result
)

print(json.dumps(
    incorrect_reflection.model_dump(),
    indent=4,
    ensure_ascii=False
))

{
    "status": "needs_correction",
    "evidence_supported": false,
    "identified_issues": [
        "Accuracy does not match evidence."
    ],
    "correction_required": true
}


## Logging and Audit Trail

The Logging and Audit Trail component records the main activities performed by the ModelGuard Agent.

It stores execution timestamps, user requests, generated plans, tool execution results, and reflection outcomes. This information supports traceability, debugging, and future system evaluation.

The project uses JSON Lines (JSONL) to store each execution as a separate record. Raw customer data and complete tool outputs are excluded from the logs.

### Initialize Logging System

In [41]:

import json
from datetime import datetime
from pathlib import Path

LOG_DIR = PROJECT_ROOT / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_LOG_FILE = LOG_DIR / "agent_audit_log.jsonl"

def log_agent_execution(
    user_request,
    planning_result,
    react_result,
    reflection_result,
    final_response
):
    plan_data = planning_result.get("plan", [])
    tool_names = [
        step.get("tool_name")
        for step in plan_data
        if isinstance(step, dict)
    ]

    execution_records = [
        {
            "tool_name": item.get("tool_name"),
            "status": item.get("status"),
            "execution_time": item.get("execution_time")
        }
        for item in react_result.get("execution_log", [])
    ]

    reflection_data = (
        reflection_result.model_dump()
        if hasattr(reflection_result, "model_dump")
        else reflection_result
    )

    response_data = (
        final_response.model_dump()
        if hasattr(final_response, "model_dump")
        else final_response
    )

    record = {
        "timestamp": datetime.now().astimezone().isoformat(),
        "agent": AGENT_NAME,
        "model": MODEL_NAME,
        "user_request": user_request,
        "planned_tools": tool_names,
        "execution_status": react_result.get("status"),
        "iterations": react_result.get("iterations"),
        "used_tools": react_result.get("used_tools", []),
        "tool_execution": execution_records,
        "reflection": reflection_data,
        "final_response_status": response_data.get("status")
    }

    with open(AUDIT_LOG_FILE, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

    return record


print("Logging system initialized.")
print(f"Audit log: {AUDIT_LOG_FILE}")

Logging system initialized.
Audit log: C:\Users\User\ModelGuard\logs\agent_audit_log.jsonl


In [42]:

# Re-run reflection using the latest response and execution results
reflection_result = reflect_on_response(
    validated_response,
    react_result
)

# Save the execution record
audit_record = log_agent_execution(
    user_request=test_request,
    planning_result=planning_result,
    react_result=react_result,
    reflection_result=reflection_result,
    final_response=validated_response
)

# Verify the saved record
with open(AUDIT_LOG_FILE, "r", encoding="utf-8") as file:
    saved_records = [
        json.loads(line) for line in file if line.strip()
    ]

print("Audit logging test completed.")
print("Total records:", len(saved_records))
print("Latest record status:", saved_records[-1]["execution_status"])
print("Reflection status:", saved_records[-1]["reflection"]["status"])
print("Logged tools:", saved_records[-1]["used_tools"])


Audit logging test completed.
Total records: 34
Latest record status: incomplete
Reflection status: approved
Logged tools: ['get_model_information', 'get_performance_baseline']


## Agent Evaluation

This section evaluates the ModelGuard Agent through a set of controlled tests.

The evaluation focuses on tool selection, execution reliability, numerical accuracy, evidence grounding, and the ability to detect unsupported information.

The tests are designed to identify implementation limitations rather than claim production-level reliability.

In [43]:

evaluation_cases = [
    {
        "test_id": "T01",
        "category": "Tool Selection",
        "request": "What model is used for churn prediction?",
        "expected_tool": "get_model_information"
    },
    {
        "test_id": "T02",
        "category": "Performance Retrieval",
        "request": "Check the churn model performance.",
        "expected_tool": "get_performance_baseline"
    },
    {
        "test_id": "T03",
        "category": "Drift Detection",
        "request": "Has the churn model experienced data drift?",
        "expected_tool": "get_drift_analysis"
    },
    {
        "test_id": "T04",
        "category": "Governance Retrieval",
        "request": "What are the main risks identified for this AI system?",
        "expected_tool": "get_risk_register"
    }
]

print("Evaluation test cases:", len(evaluation_cases))
for case in evaluation_cases:
    print(case["test_id"], "-", case["category"])


Evaluation test cases: 4
T01 - Tool Selection
T02 - Performance Retrieval
T03 - Drift Detection
T04 - Governance Retrieval


### Evaluation Test Cases

In [44]:

tool_selection_results = []

for case in evaluation_cases:
    try:
        selection_result = select_tool(case["request"])

        # Extract the tool name from the returned dictionary
        selected_tool = (
            selection_result.get("tool_name")
            if isinstance(selection_result, dict)
            else selection_result
        )

        passed = selected_tool == case["expected_tool"]

        tool_selection_results.append({
            "test_id": case["test_id"],
            "category": case["category"],
            "expected_tool": case["expected_tool"],
            "selected_tool": selected_tool,
            "passed": passed
        })

    except Exception as error:
        tool_selection_results.append({
            "test_id": case["test_id"],
            "category": case["category"],
            "expected_tool": case["expected_tool"],
            "selected_tool": None,
            "passed": False,
            "error": str(error)
        })

tool_selection_df = pd.DataFrame(tool_selection_results)

display(tool_selection_df)

passed = int(tool_selection_df["passed"].sum())
total = len(tool_selection_df)

print(f"Tool Selection Accuracy: {passed}/{total} ({passed / total:.2%})")


,test_id,category,expected_tool,selected_tool,passed
0,T01,Tool Selection,get_model_information,get_model_information,True
1,T02,Performance Retrieval,get_performance_baseline,get_performance_baseline,True
2,T03,Drift Detection,get_drift_analysis,get_drift_analysis,True
3,T04,Governance Retrieval,get_risk_register,get_risk_register,True


Tool Selection Accuracy: 4/4 (100.00%)


### Tool Selection

#### Execution Reliability Test

In [45]:

execution_results = []

for case in evaluation_cases:
    tool_name = case["expected_tool"]

    try:
        result = execute_tool(tool_name)
        execution_results.append({
            "test_id": case["test_id"],
            "tool": tool_name,
            "status": result.get("status"),
            "passed": result.get("status") == "success",
            "execution_time": result.get("execution_time")
        })
    except Exception as error:
        execution_results.append({
            "test_id": case["test_id"],
            "tool": tool_name,
            "status": "error",
            "passed": False,
            "execution_time": None,
            "error": str(error)
        })

execution_df = pd.DataFrame(execution_results)
display(execution_df)

successful = int(execution_df["passed"].sum())
total = len(execution_df)

print(f"Execution Success Rate: {successful}/{total} ({successful / total:.2%})")


,test_id,tool,status,passed,execution_time
0,T01,get_model_information,success,True,0.0017
1,T02,get_performance_baseline,success,True,0.0012
2,T03,get_drift_analysis,success,True,0.3413
3,T04,get_risk_register,success,True,0.0019


Execution Success Rate: 4/4 (100.00%)


### Integrated Agent Reliability and Hallucination Evaluation

This evaluation tests the ModelGuard Agent's ability to prevent unsupported or incorrect information from reaching the final response.

Three controlled scenarios are evaluated:

1. Unsupported Claim: Detects claims that contradict verified system information.
2. Incorrect Numerical Claim: Detects numerical values that do not match tool results.
3. Insufficient Evidence: Checks whether the agent avoids making confident conclusions when required evidence is unavailable.

The evaluation uses deterministic validation rules and verified tool outputs.

In [46]:

# Response Validation Function

def evaluate_agent_response(
    response,
    evidence,
    required_evidence=None
):
    issues = []

    # Combine response text for claim checking
    response_text = " ".join(
        [response.answer]
        + response.key_findings
        + response.limitations
    ).lower()

    # Check deployment claims
    actual_status = str(
        evidence.get("deployment_status", "Unknown")
    ).lower()

    if (
        "deployed in production" in response_text
        and actual_status != "deployed"
    ):
        issues.append("Unsupported deployment claim")

    # Check automatic retraining claims
    if "automatically retrains" in response_text:
        issues.append("Unsupported automatic retraining claim")

    # Check required evidence availability
    if required_evidence:
        for item in required_evidence:
            if item not in evidence:
                uncertainty_phrases = [
                    "insufficient evidence",
                    "cannot determine",
                    "not available",
                    "unknown"
                ]

                if not any(
                    phrase in response_text
                    for phrase in uncertainty_phrases
                ):
                    issues.append(
                        f"Insufficient evidence for: {item}"
                    )

    return {
        "passed": len(issues) == 0,
        "issues": issues
    }


# Retrieve the actual deployment status
system_profile = get_system_profile()

deployment_status = system_profile.get(
    "deployment_status",
    "Unknown"
)

print("Response validation function created.")
print("Deployment Status:", deployment_status)


Response validation function created.
Deployment Status: Not yet deployed


### Create Controlled Reliability Tests

In [47]:

# Create controlled reliability test scenarios

# 1. Unsupported Claims Test
unsupported_response = validated_response.model_copy(deep=True)

unsupported_response.answer = (
    "The model is deployed in production and automatically "
    "retrains whenever data drift is detected."
)

unsupported_test = evaluate_agent_response(
    response=unsupported_response,
    evidence={"deployment_status": deployment_status}
)


# 2. Incorrect Numerical Claim Test
incorrect_numeric_response = validated_response.model_copy(deep=True)

incorrect_numeric_response.key_findings[0] = "Accuracy: 0.9500"

numeric_reflection = reflect_on_response(
    incorrect_numeric_response,
    react_result
)

numeric_test_passed = any(
    "Accuracy does not match evidence." in issue
    for issue in numeric_reflection.identified_issues
)


# 3. Insufficient Evidence Test
insufficient_evidence_response = validated_response.model_copy(deep=True)

insufficient_evidence_response.answer = (
    "The monitoring system confirms that no data drift exists."
)

insufficient_test = evaluate_agent_response(
    response=insufficient_evidence_response,
    evidence={"deployment_status": deployment_status},
    required_evidence=["drift_analysis"]
)


# Display test preparation results
print("Controlled Reliability Tests Prepared")
print("--------------------------------------")

print("Unsupported Claim Issues:")
print(unsupported_test["issues"])

print("\nIncorrect Numerical Claim Issues:")
print(numeric_reflection.identified_issues)

print("\nInsufficient Evidence Issues:")
print(insufficient_test["issues"])


Controlled Reliability Tests Prepared
--------------------------------------
Unsupported Claim Issues:
['Unsupported deployment claim', 'Unsupported automatic retraining claim']

Incorrect Numerical Claim Issues:
['Accuracy does not match evidence.']

Insufficient Evidence Issues:
['Insufficient evidence for: drift_analysis']


In [48]:

# Evaluate whether each test detected the expected problem

agent_evaluation_results = [
    {
        "test": "Unsupported Claim",
        "passed": (
            any("Unsupported deployment claim" in issue
                for issue in unsupported_test["issues"])
            and
            any("Unsupported automatic retraining claim" in issue
                for issue in unsupported_test["issues"])
        ),
        "details": unsupported_test["issues"]
    },
    {
        "test": "Incorrect Numerical Claim",
        "passed": numeric_test_passed,
        "details": numeric_reflection.identified_issues
    },
    {
        "test": "Insufficient Evidence",
        "passed": any(
            "insufficient evidence" in issue.lower()
            for issue in insufficient_test["issues"]
        ),
        "details": insufficient_test["issues"]
    }
]

agent_evaluation_df = pd.DataFrame(agent_evaluation_results)

display(agent_evaluation_df)

passed_tests = int(agent_evaluation_df["passed"].sum())
total_tests = len(agent_evaluation_df)

print(f"Agent Reliability Tests: {passed_tests}/{total_tests}")
print(f"Test Pass Rate: {passed_tests / total_tests:.2%}")

,test,passed,details
0,Unsupported Claim,True,"[Unsupported deployment claim, Unsupported aut..."
1,Incorrect Numerical Claim,True,[Accuracy does not match evidence.]
2,Insufficient Evidence,True,[Insufficient evidence for: drift_analysis]


Agent Reliability Tests: 3/3
Test Pass Rate: 100.00%


### Response Validation and Release Control

In [49]:

def release_agent_response(
    response,
    react_result,
    evidence,
    required_evidence=None
):
    issues = []

    # 1. Check factual claims and evidence availability
    factual_check = evaluate_agent_response(
        response,
        evidence,
        required_evidence
    )
    issues.extend(factual_check["issues"])

    # 2. Validate numerical claims against tool results
    reflection = reflect_on_response(
        response,
        react_result
    )
    issues.extend(reflection.identified_issues)

    # 3. Remove duplicate issues
    issues = list(dict.fromkeys(issues))

    # 4. Block invalid responses
    if issues:
        return {
            "released": False,
            "response": None,
            "message": (
                "The response was blocked because it failed "
                "the validation checks."
            ),
            "issues": issues
        }

    # 5. Release only validated responses
    return {
        "released": True,
        "response": response.model_dump(),
        "message": "The response passed validation and is approved for delivery.",
        "issues": []
    }


# Test the response release gate

test_cases = [
    {
        "name": "Unsupported Claim",
        "response": unsupported_response,
        "evidence": {"deployment_status": deployment_status},
        "required_evidence": None,
        "expected_release": False
    },
    {
        "name": "Incorrect Numerical Claim",
        "response": incorrect_numeric_response,
        "evidence": {"deployment_status": deployment_status},
        "required_evidence": None,
        "expected_release": False
    },
    {
        "name": "Insufficient Evidence",
        "response": insufficient_evidence_response,
        "evidence": {"deployment_status": deployment_status},
        "required_evidence": ["drift_analysis"],
        "expected_release": False
    },
    {
        "name": "Valid Response",
        "response": validated_response,
        "evidence": {"deployment_status": deployment_status},
        "required_evidence": None,
        "expected_release": True
    }
]

release_results = []

for case in test_cases:
    result = release_agent_response(
        response=case["response"],
        react_result=react_result,
        evidence=case["evidence"],
        required_evidence=case["required_evidence"]
    )

    release_results.append({
        "test": case["name"],
        "expected_release": case["expected_release"],
        "actual_release": result["released"],
        "passed": result["released"] == case["expected_release"],
        "issues": result["issues"]
    })

release_df = pd.DataFrame(release_results)
display(release_df)

passed = int(release_df["passed"].sum())
total = len(release_df)

print(f"Response Gate Tests: {passed}/{total}")
print(f"Gate Test Pass Rate: {passed / total:.2%}")

,test,expected_release,actual_release,passed,issues
0,Unsupported Claim,False,False,True,"[Unsupported deployment claim, Unsupported aut..."
1,Incorrect Numerical Claim,False,False,True,[Accuracy does not match evidence.]
2,Insufficient Evidence,False,False,True,[Insufficient evidence for: drift_analysis]
3,Valid Response,True,True,True,[]


Response Gate Tests: 4/4
Gate Test Pass Rate: 100.00%


### Final Agent Evaluation and Report Generation


This step combines the results of all agent evaluation tests into a final report.

It summarizes tool selection, tool execution, reliability, and response validation performance. The results are saved in JSON and CSV formats for documentation and future analysis.

In [50]:
# Build the final evaluation report

final_agent_report = {
    "project": "ModelGuard",
    "evaluation_date": datetime.now().astimezone().isoformat(),

    "results": {
        "tool_selection_accuracy": float(tool_selection_df["passed"].mean()),
        "tool_execution_success_rate": float(execution_df["passed"].mean()),

        "reliability_tests_passed": passed_tests,
        "reliability_tests_total": total_tests,
        "reliability_test_pass_rate": passed_tests / total_tests,

        "response_gate_passed": int(release_df["passed"].sum()),
        "response_gate_total": len(release_df),
        "response_gate_pass_rate": float(release_df["passed"].mean())
    },

    "reliability_tests": agent_evaluation_results,
    "response_gate_tests": release_df.to_dict(orient="records"),

    "limitations": [
        "Evaluation used a limited number of controlled test cases.",
        "The tests verify selected unsupported claims, not all hallucinations.",
        "The response gate has not yet been integrated into the FastAPI output pipeline.",
        "Monitoring results are based on a held-out test dataset, not live production data.",
        "The system has not yet been deployed."
    ]
}

# Save final JSON report
with open(
    REPORTS_DIR / "final_agent_evaluation.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(final_agent_report, file, indent=4, ensure_ascii=False)

# Save detailed test results
agent_evaluation_df.to_csv(
    REPORTS_DIR / "agent_reliability_tests.csv",
    index=False
)

release_df.to_csv(
    REPORTS_DIR / "response_gate_tests.csv",
    index=False
)

print("Final Agent Evaluation saved successfully.")
print("Tool Selection Accuracy:", f'{final_agent_report["results"]["tool_selection_accuracy"]:.2%}')
print("Tool Execution Success Rate:", f'{final_agent_report["results"]["tool_execution_success_rate"]:.2%}')
print("Reliability Test Pass Rate:", f'{final_agent_report["results"]["reliability_test_pass_rate"]:.2%}')
print("Response Gate Pass Rate:", f'{final_agent_report["results"]["response_gate_pass_rate"]:.2%}')
print("All evaluation reports saved in:", REPORTS_DIR)

Final Agent Evaluation saved successfully.
Tool Selection Accuracy: 100.00%
Tool Execution Success Rate: 100.00%
Reliability Test Pass Rate: 100.00%
Response Gate Pass Rate: 100.00%
All evaluation reports saved in: C:\Users\User\ModelGuard\reports



## Environment Setup and Project File Inventory

This section initializes the project environment and scans the existing project directories.

It generates an inventory of the files created in the previous stages, including machine learning models, monitoring reports, governance documents, agent evaluation results, memory, and audit logs.

This step provides an overview of the available project resources before proceeding with the system integration.


In [51]:

import json
import joblib
import pandas as pd
import numpy as np
import requests

from pathlib import Path
from datetime import datetime

# Initialize project directories
PROJECT_ROOT = (
    Path.cwd().parent
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MONITORING_DIR = PROJECT_ROOT / "monitoring"
GOVERNANCE_DIR = PROJECT_ROOT / "governance"
REPORTS_DIR = PROJECT_ROOT / "reports"
MEMORY_DIR = PROJECT_ROOT / "memory"
LOG_DIR = PROJECT_ROOT / "logs"

# Project directories to scan
folders_to_check = [
    "artifacts",
    "monitoring",
    "governance",
    "reports",
    "memory",
    "logs",
    "models",
    "notebooks",
    "mlruns"
]

# Generate project file inventory
project_inventory = []

for folder_name in folders_to_check:
    folder_path = PROJECT_ROOT / folder_name

    if folder_path.exists():
        for file_path in sorted(folder_path.rglob("*")):
            if (
                file_path.is_file()
                and ".ipynb_checkpoints" not in str(file_path)
            ):
                project_inventory.append({
                    "folder": folder_name,
                    "file": str(file_path.relative_to(PROJECT_ROOT)),
                    "size_kb": round(
                        file_path.stat().st_size / 1024, 2
                    )
                })

inventory_df = pd.DataFrame(project_inventory)

print("Project Root:", PROJECT_ROOT)
print("Total Project Files:", len(inventory_df))

display(inventory_df)

# Summarize files by directory
print("\nFiles by Folder:")

display(
    inventory_df.groupby("folder")
    .agg(
        file_count=("file", "count"),
        total_size_kb=("size_kb", "sum")
    )
)

# Check MLflow directories
print("\nMLflow Directory Verification:")

for folder in ["mlruns", "governance/mlruns"]:
    path = PROJECT_ROOT / folder

    if path.exists():
        print(f"{folder}: Available")
        print(f"Items: {len(list(path.iterdir()))}")
    else:
        print(f"{folder}: Not found")


Project Root: C:\Users\User\ModelGuard
Total Project Files: 55


,folder,file,size_kb
0,artifacts,artifacts\final_xgboost_model.joblib,240.69
1,artifacts,artifacts\logistic_regression.joblib,7.36
2,artifacts,artifacts\model_results.json,4.03
3,monitoring,monitoring\batch_performance_report.csv,0.43
4,monitoring,monitoring\comparison_data.csv,171.99
5,monitoring,monitoring\comparison_target.csv,4.13
6,monitoring,monitoring\performance_baseline.json,0.51
7,monitoring,monitoring\reference_data.csv,686.13
8,governance,governance\ai_management_structure.csv,1.03
9,governance,governance\ai_management_structure.json,1.66



Files by Folder:


,file_count,total_size_kb
folder,,
artifacts,3,252.08
governance,29,533.50
logs,1,28.86
memory,1,0.27
mlruns,10,502.20
monitoring,5,863.19
reports,6,6.13



MLflow Directory Verification:
mlruns: Available
Items: 1
governance/mlruns: Available
Items: 1


### Required Files Verification

In [52]:

required_files = {
    "XGBoost Model": ARTIFACTS_DIR / "final_xgboost_model.joblib",
    "Model Results": ARTIFACTS_DIR / "model_results.json",

    "Performance Baseline": MONITORING_DIR / "performance_baseline.json",
    "Batch Performance": MONITORING_DIR / "batch_performance_report.csv",
    "Reference Data": MONITORING_DIR / "reference_data.csv",
    "Comparison Data": MONITORING_DIR / "comparison_data.csv",

    "System Profile": GOVERNANCE_DIR / "ai_system_profile.json",
    "Risk Register": GOVERNANCE_DIR / "risk_register.json",
    "Governance Controls": GOVERNANCE_DIR / "lifecycle_controls.json",
    "AI Management Structure": GOVERNANCE_DIR / "ai_management_structure.json",
    "Framework Assessment": GOVERNANCE_DIR / "framework_assessment.json",
    "Responsible AI Principles": GOVERNANCE_DIR / "responsible_ai_principles.json",
    "Impact Assessment": GOVERNANCE_DIR / "impact_assessment.json",
    "Incident Management": GOVERNANCE_DIR / "incident_management.json",
    "Audit Checklist": GOVERNANCE_DIR / "audit_checklist.json",

    "Persistent Memory": MEMORY_DIR / "persistent_memory.json",
    "Agent Audit Log": LOG_DIR / "agent_audit_log.jsonl",

    "MLflow Database": PROJECT_ROOT / "mlflow.db",
    "MLflow Artifacts": PROJECT_ROOT / "mlruns"
}

verification_results = []

for name, path in required_files.items():
    verification_results.append({
        "component": name,
        "status": "Available" if path.exists() else "Missing",
        "path": str(path.relative_to(PROJECT_ROOT))
    })

verification_df = pd.DataFrame(verification_results)
display(verification_df)

print(
    "\nAvailable:",
    (verification_df["status"] == "Available").sum(),
    "/",
    len(verification_df)
)


,component,status,path
0,XGBoost Model,Available,artifacts\final_xgboost_model.joblib
1,Model Results,Available,artifacts\model_results.json
2,Performance Baseline,Available,monitoring\performance_baseline.json
3,Batch Performance,Available,monitoring\batch_performance_report.csv
4,Reference Data,Available,monitoring\reference_data.csv
5,Comparison Data,Available,monitoring\comparison_data.csv
6,System Profile,Available,governance\ai_system_profile.json
7,Risk Register,Available,governance\risk_register.json
8,Governance Controls,Available,governance\lifecycle_controls.json
9,AI Management Structure,Available,governance\ai_management_structure.json



Available: 19 / 19


### Load and Inspect the Trained Model
The trained XGBoost model will be loaded from the artifacts directory. Its structure and prediction interface will be inspected before integrating it into the agent.

This step ensures that the original preprocessing pipeline and model configuration are preserved.

In [53]:
# Load the trained XGBoost model
xgb_model = joblib.load(ARTIFACTS_DIR / "final_xgboost_model.joblib")

# Inspect the model structure
print("Model Type:", type(xgb_model))
print("Has predict method:", hasattr(xgb_model, "predict"))
print("Has predict_proba method:", hasattr(xgb_model, "predict_proba"))

# Inspect expected input features
if hasattr(xgb_model, "feature_names_in_"):
    print("\nExpected Features:")
    print(list(xgb_model.feature_names_in_))
elif hasattr(xgb_model, "named_steps"):
    print("\nPipeline Steps:")
    print(list(xgb_model.named_steps.keys()))

Model Type: <class 'sklearn.pipeline.Pipeline'>
Has predict method: True
Has predict_proba method: True

Expected Features:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']


### Create the Customer Churn Prediction Tool
A new prediction tool will be added to ModelGuard. It will allow the agent to evaluate customer records using the trained XGBoost pipeline.

The tool will validate the input features, generate a churn score, and return the prediction based on the selected decision threshold.

In [54]:
# Define the expected input features
EXPECTED_FEATURES = list(xgb_model.feature_names_in_)

CHURN_THRESHOLD = 0.56

def predict_customer_churn(customer_data):
    """
    Predict customer churn using the trained XGBoost pipeline.
    """

    # Convert input to DataFrame
    if isinstance(customer_data, dict):
        customer_data = pd.DataFrame([customer_data])

    # Validate input type
    if not isinstance(customer_data, pd.DataFrame):
        raise TypeError("Input must be a dictionary or pandas DataFrame.")

    # Validate required features
    missing_features = [
        feature for feature in EXPECTED_FEATURES
        if feature not in customer_data.columns
    ]

    extra_features = [
        feature for feature in customer_data.columns
        if feature not in EXPECTED_FEATURES
    ]

    if missing_features or extra_features:
        raise ValueError({
            "missing_features": missing_features,
            "unexpected_features": extra_features
        })

    # Preserve the original feature order
    customer_data = customer_data[EXPECTED_FEATURES].copy()

    # Generate model score
    churn_score = float(
        xgb_model.predict_proba(customer_data)[0][1]
    )

    # Apply the selected threshold
    prediction = int(churn_score >= CHURN_THRESHOLD)

    return {
        "prediction": prediction,
        "prediction_label": (
            "Churn" if prediction == 1 else "No Churn"
        ),
        "churn_score": round(churn_score, 4),
        "decision_threshold": CHURN_THRESHOLD
    }


### Loading a Customer Sample for Prediction

The test dataset was previously saved in the monitoring directory.

We load the saved comparison dataset and select one customer record to test the prediction tool without repeating the model training process.


In [55]:

# Load the saved comparison dataset
comparison_data_path = PROJECT_ROOT / "monitoring" / "comparison_data.csv"

comparison_df = pd.read_csv(comparison_data_path)

print("Dataset Shape:", comparison_df.shape)
print("\nDataset Columns:")
print(comparison_df.columns.tolist())

# Select one customer record
customer_sample = comparison_df.iloc[0].to_dict()

# Test the prediction tool
test_result = predict_customer_churn(customer_sample)

print("\nPrediction Tool Test:")
print(test_result)


Dataset Shape: (1409, 19)

Dataset Columns:
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges']

Prediction Tool Test:
{'prediction': 0, 'prediction_label': 'No Churn', 'churn_score': 0.028, 'decision_threshold': 0.56}


### Agent Tools Integration
This section integrates the customer churn prediction tool with the existing ModelGuard agent tools.

The integration will support both tools that require no input and tools that require structured input, such as customer prediction.

### Check Existing Agent Components

In [56]:

# Check all existing ModelGuard agent components

agent_components = [
    # Tool Registry
    "TOOL_REGISTRY",
    "TOOL_DESCRIPTIONS",

    # Monitoring Tools
    "get_model_information",
    "get_performance_baseline",
    "get_batch_performance",
    "get_drift_analysis",

    # Governance Tools
    "get_system_profile",
    "get_risk_register",
    "get_impact_assessment",
    "get_incident_management",
    "get_governance_controls",

    # MLflow Tools
    "get_mlflow_client",
    "get_mlflow_experiments",
    "get_mlflow_run_details",

    # System Tool
    "get_system_status",

    # Agent Reasoning and Execution
    "select_tool",
    "execute_tool",
    "generate_tool_response",
    "reason_and_plan",
    "run_react_agent",

    # Memory
    "load_persistent_memory",
    "save_persistent_memory",
    "add_short_term_memory",
    "add_persistent_memory",
    "retrieve_memory",
    "reason_and_plan_with_memory",

    # Validation and Planning
    "validate_planning_tools",
    "validate_agent_output",
    "generate_validated_plan",

    # Reflection and Response Validation
    "reflect_on_response",
    "evaluate_agent_response",
    "release_agent_response",

    # Logging
    "log_agent_execution"
]

available = [
    component for component in agent_components
    if component in globals()
]

missing = [
    component for component in agent_components
    if component not in globals()
]

print(f"Total Components: {len(agent_components)}")
print(f"Available: {len(available)}")
print(f"Missing: {len(missing)}")

print("\nMissing Components:")
for component in missing:
    print("-", component)


Total Components: 33
Available: 33
Missing: 0

Missing Components:



## Update Tool Execution Mechanism

### Parameterized Tool Execution

The existing execution mechanism supports tools without input parameters. However, the customer churn prediction tool requires structured customer data.

The execution mechanism will be extended to support both parameter-free and parameterized tools. Input validation will be performed before execution, while the existing error-handling and execution-time tracking will be preserved.


In [57]:
import inspect
import time

def execute_tool(tool_name, tool_args=None):

    if tool_name not in TOOL_REGISTRY:
        raise ValueError(f"Unknown tool: {tool_name}")

    if tool_args is None:
        tool_args = {}

    if not isinstance(tool_args, dict):
        raise TypeError("Tool arguments must be provided as a dictionary.")

    tool_function = TOOL_REGISTRY[tool_name]

    start_time = time.perf_counter()

    try:
        # Validate arguments against the function signature
        signature = inspect.signature(tool_function)
        signature.bind(**tool_args)

        # Execute the tool with the provided arguments
        result = tool_function(**tool_args)

        execution_time = time.perf_counter() - start_time

        return {
            "tool_name": tool_name,
            "status": "success",
            "execution_time": round(execution_time, 4),
            "result": result
        }

    except Exception as error:
        return {
            "tool_name": tool_name,
            "status": "error",
            "execution_time": round(
                time.perf_counter() - start_time, 4
            ),
            "error": str(error)
        }


In [58]:

# Register the customer churn prediction tool

TOOL_REGISTRY["predict_customer_churn"] = predict_customer_churn

TOOL_DESCRIPTIONS["predict_customer_churn"] = (
    "Predict customer churn using the trained XGBoost model. "
    "Requires structured customer data containing all 19 features."
)

PLANNING_TOOLS["predict_customer_churn"] = (
    TOOL_DESCRIPTIONS["predict_customer_churn"]
)

print("Prediction Tool Registered Successfully")
print("Registered Tools:", len(TOOL_REGISTRY))
print("Tool Available:", "predict_customer_churn" in TOOL_REGISTRY)


Prediction Tool Registered Successfully
Registered Tools: 13
Tool Available: True


### Register the Churn Prediction Tool

In [59]:

# Test 1: Execute a tool without input parameters
drift_result = execute_tool("get_drift_analysis")

# Test 2: Execute a tool with structured input parameters
prediction_result = execute_tool(
    "predict_customer_churn",
    tool_args={"customer_data": customer_sample}
)

# Display test results
execution_tests = pd.DataFrame([
    {
        "Tool": drift_result["tool_name"],
        "Status": drift_result["status"],
        "Execution Time": drift_result["execution_time"]
    },
    {
        "Tool": prediction_result["tool_name"],
        "Status": prediction_result["status"],
        "Execution Time": prediction_result["execution_time"]
    }
])

display(execution_tests)

print("\nPrediction Result:")
print(prediction_result.get("result", prediction_result.get("error")))


,Tool,Status,Execution Time
0,get_drift_analysis,success,0.1711
1,predict_customer_churn,success,0.0254



Prediction Result:
{'prediction': 0, 'prediction_label': 'No Churn', 'churn_score': 0.028, 'decision_threshold': 0.56}


### Agent Tool Selection Integration

After registering the customer churn prediction tool, the agent must be able to identify it based on the user's request.

This section tests whether the tool selection component can correctly select the prediction tool from the available tools.


In [60]:

# Test the selection of the customer churn prediction tool

test_request = (
    "Predict whether this customer is likely to churn."
)

selection = select_tool(test_request)

print("User Request:", test_request)
print("Selected Tool:", selection["tool_name"])


User Request: Predict whether this customer is likely to churn.
Selected Tool: predict_customer_churn



### Prediction Tool Execution Through the Agent

After verifying that the agent can select the prediction tool, the next step is to test its execution through the agent's tool execution mechanism.

The test will pass structured customer data to the selected tool and verify that the agent receives the prediction result successfully.


In [61]:

# Execute the selected prediction tool with customer data

agent_tool_result = execute_tool(
    tool_name=selection["tool_name"],
    tool_args={
        "customer_data": customer_sample
    }
)

print("Execution Status:", agent_tool_result["status"])
print("Executed Tool:", agent_tool_result["tool_name"])

if agent_tool_result["status"] == "success":
    print("\nPrediction Result:")
    print(json.dumps(
        agent_tool_result["result"],
        indent=2
    ))
else:
    print("\nExecution Error:", agent_tool_result["error"])


Execution Status: success
Executed Tool: predict_customer_churn

Prediction Result:
{
  "prediction": 0,
  "prediction_label": "No Churn",
  "churn_score": 0.028,
  "decision_threshold": 0.56
}



### Update ReAct Agent for Structured Tool Inputs

The existing ReAct agent executes tools according to a predefined execution plan. However, the customer churn prediction tool requires structured customer data.

The ReAct execution mechanism will be extended to pass input parameters to tools when required. It will also generate responses based on the available evidence, allowing prediction tasks and monitoring tasks to be handled independently.

This update preserves the existing execution logs, observations, and validation mechanisms.


In [62]:

def run_react_agent(user_request, execution_plan, tool_inputs=None):

    observations = []
    execution_log = []
    used_tools = []

    if tool_inputs is None:
        tool_inputs = {}

    if not isinstance(tool_inputs, dict):
        return {
            "status": "error",
            "error": "Tool inputs must be provided as a dictionary.",
            "iterations": 0,
            "used_tools": [],
            "observations": [],
            "execution_log": [],
            "final_response": None
        }

    # 1. Validate the execution plan
    if not isinstance(execution_plan, list) or not execution_plan:
        return {
            "status": "error",
            "error": "The execution plan is empty or invalid.",
            "iterations": 0,
            "used_tools": [],
            "observations": [],
            "execution_log": [],
            "final_response": None
        }

    planned_tools = []

    for step in execution_plan:

        if not isinstance(step, dict):
            return {
                "status": "error",
                "error": "Invalid execution plan step."
            }

        tool_name = step.get("tool_name")

        if tool_name not in TOOL_REGISTRY:
            return {
                "status": "error",
                "error": f"Unknown tool in plan: {tool_name}"
            }

        if tool_name in planned_tools:
            return {
                "status": "error",
                "error": f"Repeated tool in plan: {tool_name}"
            }

        planned_tools.append(tool_name)

    # 2. Execute tools with their required inputs
    for iteration, step in enumerate(execution_plan, start=1):

        tool_name = step["tool_name"]

        tool_args = tool_inputs.get(tool_name, {})

        tool_result = execute_tool(
            tool_name,
            tool_args=tool_args
        )

        success = tool_result.get("status") == "success"

        used_tools.append(tool_name)

        execution_log.append({
            "iteration": iteration,
            "tool_name": tool_name,
            "purpose": step.get("purpose", ""),
            "execution_status": tool_result.get("status"),
            "execution_time": tool_result.get("execution_time")
        })

        if not success:
            return {
                "status": "error",
                "error": tool_result.get("error"),
                "iterations": iteration,
                "used_tools": used_tools,
                "observations": observations,
                "execution_log": execution_log,
                "final_response": None
            }

        observations.append({
            "iteration": iteration,
            "tool_name": tool_name,
            "status": "success",
            "result": tool_result.get("result")
        })

    # 3. Collect evidence from observations
    evidence = {
        item["tool_name"]: item["result"]
        for item in observations
    }

    # Store generated reports
    response_sections = []

    # 4. Generate a prediction response when prediction evidence exists
    prediction = evidence.get("predict_customer_churn")

    if prediction is not None:

        final_response = f"""
### Customer Churn Prediction

**Prediction:** {prediction["prediction_label"]}

**Churn Score:** {prediction["churn_score"]:.4f}

**Decision Threshold:** {prediction["decision_threshold"]:.2f}

#### Interpretation

The XGBoost model classified the customer as
{"likely to churn" if prediction["prediction"] == 1 else "not likely to churn"}
based on the selected decision threshold.

The churn score represents the model's estimated
probability score and is not necessarily calibrated.

#### Limitation

This prediction is based on the supplied customer
record and the trained model. It should support,
not replace, business judgment.
"""

        response_sections.append(final_response)

    # 5. Generate a monitoring response when performance
    # and drift evidence are both available
    performance = evidence.get("get_performance_baseline")
    drift = evidence.get("get_drift_analysis")

    if performance is not None and drift is not None:

        metrics = performance.get("metrics")

        required_metrics = [
            "Accuracy", "Precision", "Recall",
            "F1-Score", "ROC-AUC", "PR-AUC"
        ]

        if not isinstance(metrics, dict):
            return {
                "status": "incomplete",
                "error": "Performance metrics are missing or invalid.",
                "iterations": len(execution_log),
                "used_tools": used_tools,
                "observations": observations,
                "execution_log": execution_log,
                "evidence": evidence,
                "final_response": None
            }

        missing_metrics = [
            metric for metric in required_metrics
            if metric not in metrics
        ]

        if missing_metrics:
            return {
                "status": "incomplete",
                "error": f"Missing metrics: {missing_metrics}",
                "iterations": len(execution_log),
                "used_tools": used_tools,
                "observations": observations,
                "execution_log": execution_log,
                "evidence": evidence,
                "final_response": None
            }

        for metric in required_metrics:

            value = metrics[metric]

            if (
                not isinstance(value, (int, float))
                or isinstance(value, bool)
                or not 0 <= value <= 1
            ):
                return {
                    "status": "incomplete",
                    "error": f"Invalid metric value: {metric}",
                    "iterations": len(execution_log),
                    "used_tools": used_tools,
                    "observations": observations,
                    "execution_log": execution_log,
                    "evidence": evidence,
                    "final_response": None
                }

        number_of_features = drift.get("number_of_features")
        drifted_features = drift.get("drifted_features")

        if (
            not isinstance(number_of_features, int)
            or isinstance(number_of_features, bool)
            or number_of_features < 1
            or not isinstance(drifted_features, list)
            or len(drifted_features) > number_of_features
        ):
            return {
                "status": "incomplete",
                "error": "Drift analysis results are invalid.",
                "iterations": len(execution_log),
                "used_tools": used_tools,
                "observations": observations,
                "execution_log": execution_log,
                "evidence": evidence,
                "final_response": None
            }

        metric_text = "\n".join(
            f"- **{name}:** {metrics[name]:.4f}"
            for name in required_metrics
        )

        drift_text = (
            "Statistically significant drift was detected in: "
            + ", ".join(drifted_features) + "."
            if drifted_features
            else (
                f"No statistically significant drift was detected "
                f"across the {number_of_features} analyzed features."
            )
        )

        monitoring_response = f"""
### Churn Prediction Model Assessment

**Model:** {performance.get("model", "Not specified")}

**Evaluation Dataset:** {performance.get("evaluation_dataset", "Not specified")}

**Classification Threshold:** {performance.get("threshold", "Not specified")}

#### Baseline Performance Metrics

{metric_text}

#### Drift Analysis

**Analysis Type:** {drift.get("analysis_type", "Not specified")}

**Number of Analyzed Features:** {number_of_features}

**Number of Drifted Features:** {len(drifted_features)}

{drift_text}

#### Limitations

The drift analysis compares reference and comparison
datasets. It does not represent live production monitoring.
"""

        response_sections.append(monitoring_response)

    # 6. Combine available reports without inventing evidence
    if response_sections:

        final_response = "\n\n".join(response_sections)

    else:

        final_response = (
            "The requested tools were executed successfully. "
            "However, the available evidence is not sufficient "
            "to generate a specialized report."
        )

    return {
        "status": "completed",
        "iterations": len(execution_log),
        "used_tools": used_tools,
        "observations": observations,
        "execution_log": execution_log,
        "evidence": evidence,
        "final_response": final_response
    }


print("Updated ReAct Agent created successfully.")


Updated ReAct Agent created successfully.


### ReAct Agent Prediction Test

This test checks whether the ReAct agent can execute the customer churn prediction tool using structured customer data. The customer data is passed separately from the execution plan to ensure controlled tool execution.

In [63]:
prediction_plan = [
    {
        "step": 1,
        "tool_name": "predict_customer_churn",
        "purpose": "Predict the churn risk of the selected customer."
    }
]

prediction_agent_result = run_react_agent(
    user_request="Predict whether this customer is likely to churn.",
    execution_plan=prediction_plan,
    tool_inputs={
        "predict_customer_churn": {
            "customer_data": customer_sample
        }
    }
)

print("Agent Status:", prediction_agent_result["status"])
print("Used Tools:", prediction_agent_result["used_tools"])
print("\nFinal Response:")
print(prediction_agent_result["final_response"])

Agent Status: completed
Used Tools: ['predict_customer_churn']

Final Response:

### Customer Churn Prediction

**Prediction:** No Churn

**Churn Score:** 0.0280

**Decision Threshold:** 0.56

#### Interpretation

The XGBoost model classified the customer as
not likely to churn
based on the selected decision threshold.

The churn score represents the model's estimated
probability score and is not necessarily calibrated.

#### Limitation

This prediction is based on the supplied customer
record and the trained model. It should support,
not replace, business judgment.



### ReAct Agent Monitoring Test

This test evaluates whether the agent can retrieve model performance and drift analysis results. It checks the monitoring workflow independently from customer-level prediction.

In [64]:
monitoring_plan = [
    {
        "step": 1,
        "tool_name": "get_performance_baseline",
        "purpose": "Retrieve the model performance metrics."
    },
    {
        "step": 2,
        "tool_name": "get_drift_analysis",
        "purpose": "Retrieve the feature drift analysis."
    }
]

monitoring_agent_result = run_react_agent(
    user_request="Analyze model performance and data drift.",
    execution_plan=monitoring_plan
)

print("Agent Status:", monitoring_agent_result["status"])
print("Used Tools:", monitoring_agent_result["used_tools"])
print("\nFinal Response:")
print(monitoring_agent_result["final_response"])

Agent Status: completed
Used Tools: ['get_performance_baseline', 'get_drift_analysis']

Final Response:

### Churn Prediction Model Assessment

**Model:** XGBoost

**Evaluation Dataset:** Held-out Test Dataset

**Classification Threshold:** 0.56

#### Baseline Performance Metrics

- **Accuracy:** 0.7644
- **Precision:** 0.5407
- **Recall:** 0.7460
- **F1-Score:** 0.6270
- **ROC-AUC:** 0.8463
- **PR-AUC:** 0.6622

#### Drift Analysis

**Analysis Type:** Reference vs Comparison Data

**Number of Analyzed Features:** 19

**Number of Drifted Features:** 0

No statistically significant drift was detected across the 19 analyzed features.

#### Limitations

The drift analysis compares reference and comparison
datasets. It does not represent live production monitoring.



### ReAct Agent Combined Workflow Test

This test evaluates whether the agent can execute multiple tools within a single workflow. It combines customer churn prediction with model performance and drift analysis.

In [65]:
combined_plan = [
    {
        "step": 1,
        "tool_name": "predict_customer_churn",
        "purpose": "Predict the churn risk of the selected customer."
    },
    {
        "step": 2,
        "tool_name": "get_performance_baseline",
        "purpose": "Retrieve the model performance metrics."
    },
    {
        "step": 3,
        "tool_name": "get_drift_analysis",
        "purpose": "Retrieve the feature drift analysis."
    }
]

combined_agent_result = run_react_agent(
    user_request=(
        "Predict this customer's churn risk and assess "
        "the model's performance and data drift."
    ),
    execution_plan=combined_plan,
    tool_inputs={
        "predict_customer_churn": {
            "customer_data": customer_sample
        }
    }
)

print("Agent Status:", combined_agent_result["status"])
print("Used Tools:", combined_agent_result["used_tools"])
print("\nFinal Response:")
print(combined_agent_result["final_response"])

Agent Status: completed
Used Tools: ['predict_customer_churn', 'get_performance_baseline', 'get_drift_analysis']

Final Response:

### Customer Churn Prediction

**Prediction:** No Churn

**Churn Score:** 0.0280

**Decision Threshold:** 0.56

#### Interpretation

The XGBoost model classified the customer as
not likely to churn
based on the selected decision threshold.

The churn score represents the model's estimated
probability score and is not necessarily calibrated.

#### Limitation

This prediction is based on the supplied customer
record and the trained model. It should support,
not replace, business judgment.



### Churn Prediction Model Assessment

**Model:** XGBoost

**Evaluation Dataset:** Held-out Test Dataset

**Classification Threshold:** 0.56

#### Baseline Performance Metrics

- **Accuracy:** 0.7644
- **Precision:** 0.5407
- **Recall:** 0.7460
- **F1-Score:** 0.6270
- **ROC-AUC:** 0.8463
- **PR-AUC:** 0.6622

#### Drift Analysis

**Analysis Type:** Reference vs Compariso

### ReAct Agent Prediction-Only Validation

This test verifies that the agent generates only the customer prediction report when the execution plan contains the prediction tool.

In [66]:
prediction_only_result = run_react_agent(
    user_request="Predict whether this customer is likely to churn.",
    execution_plan=prediction_plan,
    tool_inputs={
        "predict_customer_churn": {
            "customer_data": customer_sample
        }
    }
)

print("Status:", prediction_only_result["status"])
print("Used Tools:", prediction_only_result["used_tools"])
print("\nFinal Response:")
print(prediction_only_result["final_response"])

assert prediction_only_result["status"] == "completed"
assert "Customer Churn Prediction" in prediction_only_result["final_response"]
assert "Baseline Performance Metrics" not in prediction_only_result["final_response"]

print("\nPrediction-only validation: PASSED")

Status: completed
Used Tools: ['predict_customer_churn']

Final Response:

### Customer Churn Prediction

**Prediction:** No Churn

**Churn Score:** 0.0280

**Decision Threshold:** 0.56

#### Interpretation

The XGBoost model classified the customer as
not likely to churn
based on the selected decision threshold.

The churn score represents the model's estimated
probability score and is not necessarily calibrated.

#### Limitation

This prediction is based on the supplied customer
record and the trained model. It should support,
not replace, business judgment.


Prediction-only validation: PASSED


### ReAct Agent Monitoring-Only Validation

This test verifies that the agent generates a monitoring report without including customer-level prediction results.

In [67]:
monitoring_only_result = run_react_agent(
    user_request="Assess model performance and data drift.",
    execution_plan=monitoring_plan
)

print("Status:", monitoring_only_result["status"])
print("Used Tools:", monitoring_only_result["used_tools"])
print("\nFinal Response:")
print(monitoring_only_result["final_response"])

assert monitoring_only_result["status"] == "completed"
assert "Baseline Performance Metrics" in monitoring_only_result["final_response"]
assert "Drift Analysis" in monitoring_only_result["final_response"]
assert "Customer Churn Prediction" not in monitoring_only_result["final_response"]

print("\nMonitoring-only validation: PASSED")

Status: completed
Used Tools: ['get_performance_baseline', 'get_drift_analysis']

Final Response:

### Churn Prediction Model Assessment

**Model:** XGBoost

**Evaluation Dataset:** Held-out Test Dataset

**Classification Threshold:** 0.56

#### Baseline Performance Metrics

- **Accuracy:** 0.7644
- **Precision:** 0.5407
- **Recall:** 0.7460
- **F1-Score:** 0.6270
- **ROC-AUC:** 0.8463
- **PR-AUC:** 0.6622

#### Drift Analysis

**Analysis Type:** Reference vs Comparison Data

**Number of Analyzed Features:** 19

**Number of Drifted Features:** 0

No statistically significant drift was detected across the 19 analyzed features.

#### Limitations

The drift analysis compares reference and comparison
datasets. It does not represent live production monitoring.


Monitoring-only validation: PASSED


### ReAct Agent Insufficient Evidence Test

This test checks whether the agent handles incomplete evidence correctly. It executes only the model information tool, which does not provide enough information to generate a complete prediction or monitoring report.

In [68]:
insufficient_evidence_plan = [
    {
        "step": 1,
        "tool_name": "get_model_information",
        "purpose": "Retrieve general model information."
    }
]

insufficient_evidence_result = run_react_agent(
    user_request="Provide a complete assessment of model performance and customer churn risk.",
    execution_plan=insufficient_evidence_plan
)

print("Status:", insufficient_evidence_result["status"])
print("Used Tools:", insufficient_evidence_result["used_tools"])
print("\nFinal Response:")
print(insufficient_evidence_result["final_response"])

assert insufficient_evidence_result["status"] == "completed"
assert "not sufficient" in insufficient_evidence_result["final_response"].lower()
assert "Accuracy" not in insufficient_evidence_result["final_response"]

print("\nInsufficient-evidence validation: PASSED")

Status: completed
Used Tools: ['get_model_information']

Final Response:
The requested tools were executed successfully. However, the available evidence is not sufficient to generate a specialized report.

Insufficient-evidence validation: PASSED


### ReAct Agent Workflow Evaluation

This section summarizes the functional tests performed on the ReAct agent. The tests evaluate prediction, monitoring, combined execution, and handling of insufficient evidence. Results are stored for reproducibility and later evaluation.

In [69]:
react_workflow_tests = [
    {
        "test_name": "Prediction Only",
        "expected_behavior": "Generate a customer prediction report only.",
        "actual_status": prediction_only_result["status"],
        "passed": (
            prediction_only_result["status"] == "completed"
            and "Customer Churn Prediction" in prediction_only_result["final_response"]
            and "Baseline Performance Metrics" not in prediction_only_result["final_response"]
        )
    },
    {
        "test_name": "Monitoring Only",
        "expected_behavior": "Generate a monitoring report only.",
        "actual_status": monitoring_only_result["status"],
        "passed": (
            monitoring_only_result["status"] == "completed"
            and "Baseline Performance Metrics" in monitoring_only_result["final_response"]
            and "Customer Churn Prediction" not in monitoring_only_result["final_response"]
        )
    },
    {
        "test_name": "Combined Workflow",
        "expected_behavior": "Generate prediction and monitoring reports.",
        "actual_status": combined_agent_result["status"],
        "passed": (
            combined_agent_result["status"] == "completed"
            and "Customer Churn Prediction" in combined_agent_result["final_response"]
            and "Baseline Performance Metrics" in combined_agent_result["final_response"]
            and "Drift Analysis" in combined_agent_result["final_response"]
        )
    },
    {
        "test_name": "Insufficient Evidence",
        "expected_behavior": "Avoid generating unsupported conclusions.",
        "actual_status": insufficient_evidence_result["status"],
        "passed": (
            insufficient_evidence_result["status"] == "completed"
            and "not sufficient" in insufficient_evidence_result["final_response"].lower()
        )
    }
]

react_workflow_report = pd.DataFrame(react_workflow_tests)

react_workflow_report["result"] = react_workflow_report["passed"].map({
    True: "Passed",
    False: "Failed"
})

display(react_workflow_report)

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

react_workflow_report.to_csv(
    REPORTS_DIR / "react_workflow_evaluation.csv",
    index=False
)

print("\nTotal Tests:", len(react_workflow_report))
print("Passed Tests:", int(react_workflow_report["passed"].sum()))
print("Failed Tests:", int((~react_workflow_report["passed"]).sum()))
print("Report saved successfully.")

,test_name,expected_behavior,actual_status,passed,result
0,Prediction Only,Generate a customer prediction report only.,completed,True,Passed
1,Monitoring Only,Generate a monitoring report only.,completed,True,Passed
2,Combined Workflow,Generate prediction and monitoring reports.,completed,True,Passed
3,Insufficient Evidence,Avoid generating unsupported conclusions.,completed,True,Passed



Total Tests: 4
Passed Tests: 4
Failed Tests: 0
Report saved successfully.


### Agent Validation Integration

This section checks the existing response validation and reflection components before integrating them with the updated ReAct workflow. The goal is to reuse the existing validation mechanisms and avoid duplicate implementations.

In [70]:
validation_components = {
    "evaluate_agent_response": "evaluate_agent_response" in globals(),
    "reflect_on_response": "reflect_on_response" in globals(),
    "release_agent_response": "release_agent_response" in globals(),
    "FinalResponse": "FinalResponse" in globals(),
    "ReflectionOutput": "ReflectionOutput" in globals(),
    "log_agent_execution": "log_agent_execution" in globals()
}

for component, available in validation_components.items():
    print(f"{component}: {'Available' if available else 'Missing'}")

print(
    "\nAvailable Components:",
    sum(validation_components.values()),
    "/",
    len(validation_components)
)

evaluate_agent_response: Available
reflect_on_response: Available
release_agent_response: Available
FinalResponse: Available
ReflectionOutput: Available
log_agent_execution: Available

Available Components: 6 / 6


### ReAct and Response Validation Integration Test

This test evaluates whether the combined ReAct output can pass through the existing response validation and reflection mechanisms. It checks whether the final response is supported by the collected evidence.

In [71]:
print("ReAct Status:", combined_agent_result["status"])

print("\nEvidence Keys:")
print(list(combined_agent_result["evidence"].keys()))

print("\nFinal Response:")
print(combined_agent_result["final_response"])

ReAct Status: completed

Evidence Keys:
['predict_customer_churn', 'get_performance_baseline', 'get_drift_analysis']

Final Response:

### Customer Churn Prediction

**Prediction:** No Churn

**Churn Score:** 0.0280

**Decision Threshold:** 0.56

#### Interpretation

The XGBoost model classified the customer as
not likely to churn
based on the selected decision threshold.

The churn score represents the model's estimated
probability score and is not necessarily calibrated.

#### Limitation

This prediction is based on the supplied customer
record and the trained model. It should support,
not replace, business judgment.



### Churn Prediction Model Assessment

**Model:** XGBoost

**Evaluation Dataset:** Held-out Test Dataset

**Classification Threshold:** 0.56

#### Baseline Performance Metrics

- **Accuracy:** 0.7644
- **Precision:** 0.5407
- **Recall:** 0.7460
- **F1-Score:** 0.6270
- **ROC-AUC:** 0.8463
- **PR-AUC:** 0.6622

#### Drift Analysis

**Analysis Type:** Reference vs Compa

### Validation Component Interface Inspection

This step inspects the function signatures and output schema fields of the existing validation components. It helps ensure that the ReAct output is passed correctly to the reflection and response validation layers.

In [72]:
import inspect

validation_functions = [
    evaluate_agent_response,
    reflect_on_response,
    release_agent_response,
    log_agent_execution
]

for function in validation_functions:
    print(f"\n{function.__name__}")
    print("Signature:", inspect.signature(function))

print("\nFinalResponse Fields:")
print(list(FinalResponse.model_fields.keys()))

print("\nReflectionOutput Fields:")
print(list(ReflectionOutput.model_fields.keys()))


evaluate_agent_response
Signature: (response, evidence, required_evidence=None)

reflect_on_response
Signature: (response_data, react_result)

release_agent_response
Signature: (response, react_result, evidence, required_evidence=None)

log_agent_execution
Signature: (user_request, planning_result, react_result, reflection_result, final_response)

FinalResponse Fields:
['status', 'answer', 'key_findings', 'evidence', 'limitations']

ReflectionOutput Fields:
['status', 'evidence_supported', 'identified_issues', 'correction_required']


### Validation Logic Inspection

This step examines the internal implementation of the existing validation functions. The purpose is to understand how evidence, reflection results, and response approval are handled before integrating them into the ReAct workflow.

In [73]:

functions_to_inspect = [
    evaluate_agent_response,
    reflect_on_response,
    release_agent_response
]

for function in functions_to_inspect:
    print("\n" + "=" * 70)
    print(f"FUNCTION: {function.__name__}")
    print("=" * 70)
    print(inspect.getsource(function))



FUNCTION: evaluate_agent_response
def evaluate_agent_response(
    response,
    evidence,
    required_evidence=None
):
    issues = []

    # Combine response text for claim checking
    response_text = " ".join(
        [response.answer]
        + response.key_findings
        + response.limitations
    ).lower()

    # Check deployment claims
    actual_status = str(
        evidence.get("deployment_status", "Unknown")
    ).lower()

    if (
        "deployed in production" in response_text
        and actual_status != "deployed"
    ):
        issues.append("Unsupported deployment claim")

    # Check automatic retraining claims
    if "automatically retrains" in response_text:
        issues.append("Unsupported automatic retraining claim")

    # Check required evidence availability
    if required_evidence:
        for item in required_evidence:
            if item not in evidence:
                uncertainty_phrases = [
                    "insufficient evidence",
           


### Reflection Enhancement and Evidence Validation

The reflection component checks whether the agent's final response is consistent with the evidence collected from the executed tools.

The validation process supports three types of evidence:

- Customer churn prediction results.
- Model performance metrics.
- Data drift analysis results.

The updated function validates only the evidence available in the current execution. This allows the agent to handle prediction, monitoring, and combined requests without requiring unrelated evidence.

The reflection component also identifies unsupported or incorrect numerical claims before the response is approved.


In [74]:

def reflect_on_response(response_data, react_result):
    """Validate response claims against available tool evidence."""

    issues = []
    supported_claims = 0
    checked_claims = 0

    evidence = react_result.get("evidence", {})

    if react_result.get("status") != "completed":
        issues.append("Agent execution was not completed.")

    def get_claim(prefix):
        for item in response_data.key_findings:
            if item.lower().startswith(prefix.lower() + ":"):
                return item.split(":", 1)[1].strip()
        return None

    def validate_text_claim(prefix, expected):
        nonlocal checked_claims, supported_claims

        reported = get_claim(prefix)

        if reported is not None:
            checked_claims += 1

            if reported.lower() == str(expected).lower():
                supported_claims += 1
            else:
                issues.append(
                    f"{prefix} does not match evidence."
                )

    def validate_numeric_claim(prefix, expected, tolerance=0.0001):
        nonlocal checked_claims, supported_claims

        reported = get_claim(prefix)

        if reported is not None:
            checked_claims += 1

            try:
                reported_value = float(reported)

                if abs(reported_value - float(expected)) <= tolerance:
                    supported_claims += 1
                else:
                    issues.append(
                        f"{prefix} does not match evidence."
                    )

            except (ValueError, TypeError):
                issues.append(
                    f"{prefix} has an invalid numerical value."
                )

    # 1. Validate customer prediction when available
    prediction = evidence.get("predict_customer_churn")

    if prediction is not None:

        validate_text_claim(
            "Prediction",
            prediction["prediction_label"]
        )

        validate_numeric_claim(
            "Churn Score",
            prediction["churn_score"]
        )

        validate_numeric_claim(
            "Decision Threshold",
            prediction["decision_threshold"]
        )

    # 2. Validate performance metrics when available
    performance = evidence.get("get_performance_baseline")

    if performance is not None:

        expected_metrics = performance.get("metrics", {})

        metric_map = {
            "Accuracy": "Accuracy",
            "Precision": "Precision",
            "Recall": "Recall",
            "F1-Score": "F1-Score",
            "ROC-AUC": "ROC-AUC",
            "PR-AUC": "PR-AUC"
        }

        for label, key in metric_map.items():

            if key in expected_metrics:
                validate_numeric_claim(
                    label,
                    expected_metrics[key]
                )

    # 3. Validate drift information when available
    drift = evidence.get("get_drift_analysis")

    if drift is not None:

        validate_numeric_claim(
            "Analyzed Features",
            drift["number_of_features"],
            tolerance=0
        )

        validate_numeric_claim(
            "Drifted Features",
            len(drift["drifted_features"]),
            tolerance=0
        )

    # 4. Check whether any evidence-based claims were validated
    if checked_claims == 0:
        issues.append(
            "No evidence-based claims were available for reflection."
        )

    evidence_supported = (
        checked_claims > 0
        and supported_claims == checked_claims
    )

    correction_required = len(issues) > 0

    return ReflectionOutput(
        status="needs_correction" if correction_required else "approved",
        evidence_supported=evidence_supported,
        identified_issues=issues,
        correction_required=correction_required
    )


print("Updated Reflection Function Created Successfully.")


Updated Reflection Function Created Successfully.



### Structured Response Generation

The ReAct agent generates a textual response based on the results returned by its tools. However, the validation components require a structured response.

This section introduces a response conversion function that transforms the ReAct output into the FinalResponse schema.

The structured response contains:

- The generated answer.
- Key findings extracted directly from tool evidence.
- Evidence sources used during execution.
- Relevant limitations.

Extracting key findings directly from tool results allows the reflection component to compare reported values with the original evidence.


In [75]:

def build_structured_response(react_result):

    evidence = react_result.get("evidence", {})

    key_findings = []
    evidence_sources = []
    limitations = []

    # 1. Extract customer prediction findings
    prediction = evidence.get("predict_customer_churn")

    if prediction is not None:

        key_findings.extend([
            f"Prediction: {prediction['prediction_label']}",
            f"Churn Score: {prediction['churn_score']}",
            f"Decision Threshold: {prediction['decision_threshold']}"
        ])

        evidence_sources.append(
            "Customer churn prediction tool"
        )

        limitations.append(
            "The churn score is not necessarily calibrated."
        )

    # 2. Extract model performance findings
    performance = evidence.get("get_performance_baseline")

    if performance is not None:

        metrics = performance.get("metrics", {})

        metric_names = [
            "Accuracy",
            "Precision",
            "Recall",
            "F1-Score",
            "ROC-AUC",
            "PR-AUC"
        ]

        for metric in metric_names:

            if metric in metrics:
                key_findings.append(
                    f"{metric}: {metrics[metric]}"
                )

        evidence_sources.append(
            "Model performance baseline"
        )

    # 3. Extract drift analysis findings
    drift = evidence.get("get_drift_analysis")

    if drift is not None:

        number_of_features = drift.get("number_of_features")
        drifted_features = drift.get("drifted_features", [])

        if number_of_features is not None:

            key_findings.append(
                f"Analyzed Features: {number_of_features}"
            )

        key_findings.append(
            f"Drifted Features: {len(drifted_features)}"
        )

        evidence_sources.append(
            "Feature drift analysis"
        )

        limitations.append(
            "Drift analysis uses reference and comparison "
            "datasets and does not represent live monitoring."
        )

    # 4. Add execution limitations
    if react_result.get("status") != "completed":

        limitations.append(
            "The agent execution was not completed successfully."
        )

    # 5. Build the structured response
    structured_response = FinalResponse(
        status=(
            "success"
            if react_result.get("status") == "completed"
            else "partial"
        ),
        answer=react_result.get("final_response") or (
            "Insufficient evidence to generate a specialized report."
        ),
        key_findings=key_findings,
        evidence=evidence_sources,
        limitations=limitations
    )

    return structured_response


print("Structured Response Builder Created Successfully.")


Structured Response Builder Created Successfully.



### Structured Response Builder Testing

This test verifies that the Structured Response Builder correctly converts the combined ReAct output into the FinalResponse schema.

The test checks whether customer prediction results, model performance metrics, and drift analysis findings are extracted from the original tool evidence.

The generated response will be inspected to ensure that its structure and key findings are consistent with the collected evidence.

In [76]:

# Convert the combined ReAct result into a structured response

structured_response = build_structured_response(
    combined_agent_result
)

# Display the structured response
print("Response Status:", structured_response.status)

print("\nAnswer:")
print(structured_response.answer)

print("\nKey Findings:")
for finding in structured_response.key_findings:
    print("-", finding)

print("\nEvidence Sources:")
for source in structured_response.evidence:
    print("-", source)

print("\nLimitations:")
for limitation in structured_response.limitations:
    print("-", limitation)

# Validate the response structure
assert isinstance(structured_response, FinalResponse)
assert len(structured_response.key_findings) > 0
assert len(structured_response.evidence) > 0

print("\nStructured Response Test: PASSED")


Response Status: success

Answer:

### Customer Churn Prediction

**Prediction:** No Churn

**Churn Score:** 0.0280

**Decision Threshold:** 0.56

#### Interpretation

The XGBoost model classified the customer as
not likely to churn
based on the selected decision threshold.

The churn score represents the model's estimated
probability score and is not necessarily calibrated.

#### Limitation

This prediction is based on the supplied customer
record and the trained model. It should support,
not replace, business judgment.



### Churn Prediction Model Assessment

**Model:** XGBoost

**Evaluation Dataset:** Held-out Test Dataset

**Classification Threshold:** 0.56

#### Baseline Performance Metrics

- **Accuracy:** 0.7644
- **Precision:** 0.5407
- **Recall:** 0.7460
- **F1-Score:** 0.6270
- **ROC-AUC:** 0.8463
- **PR-AUC:** 0.6622

#### Drift Analysis

**Analysis Type:** Reference vs Comparison Data

**Number of Analyzed Features:** 19

**Number of Drifted Features:** 0

No statistically


### Reflection Validation Testing

This test evaluates whether the reflection component can verify the structured response against the evidence collected by the ReAct agent.

It checks customer prediction results, model performance metrics, and data drift findings.

The objective is to identify unsupported or incorrect claims before the response is passed to the final validation gate.


In [77]:

reflection_result = reflect_on_response(
    structured_response,
    combined_agent_result
)

print("Reflection Status:", reflection_result.status)

print("Evidence Supported:", reflection_result.evidence_supported)

print("Correction Required:", reflection_result.correction_required)

print("\nIdentified Issues:")
for issue in reflection_result.identified_issues:
    print("-", issue)

print("\nReflection Validation Test:",
      "PASSED" if (
          reflection_result.status == "approved"
          and reflection_result.evidence_supported
          and not reflection_result.correction_required
      ) else "FAILED")


Reflection Status: approved
Evidence Supported: True
Correction Required: False

Identified Issues:

Reflection Validation Test: PASSED



### Response Validation Gate Testing

This test evaluates the final response approval mechanism.

The structured response is checked against the available tool evidence and reviewed by the reflection component.

The response is released only when all validation checks pass. Otherwise, the response is blocked to prevent unsupported information from reaching the user.


In [78]:

release_result = release_agent_response(
    response=structured_response,
    react_result=combined_agent_result,
    evidence=combined_agent_result["evidence"]
)

print("Response Released:", release_result["released"])
print("Validation Message:", release_result["message"])

print("\nValidation Issues:")
for issue in release_result["issues"]:
    print("-", issue)

print("\nResponse Gate Test:",
      "PASSED" if (
          release_result["released"] is True
          and release_result["response"] is not None
          and len(release_result["issues"]) == 0
      ) else "FAILED")


Response Released: True
Validation Message: The response passed validation and is approved for delivery.

Validation Issues:

Response Gate Test: PASSED



### Response Gate Negative Testing

This test evaluates whether the response validation gate can detect incorrect numerical claims.

An intentionally incorrect model accuracy value is introduced into the structured response. The validation system should compare this value with the original tool evidence and block the response if a mismatch is detected.


In [79]:

# Create an intentionally incorrect response
incorrect_response = structured_response.model_copy(deep=True)

incorrect_findings = []

for finding in incorrect_response.key_findings:

    if finding.startswith("Accuracy:"):
        incorrect_findings.append("Accuracy: 0.9500")
    else:
        incorrect_findings.append(finding)

incorrect_response.key_findings = incorrect_findings

# Submit the incorrect response to the validation gate
negative_release_result = release_agent_response(
    response=incorrect_response,
    react_result=combined_agent_result,
    evidence=combined_agent_result["evidence"]
)

print("Response Released:", negative_release_result["released"])
print("Validation Message:", negative_release_result["message"])

print("\nDetected Issues:")
for issue in negative_release_result["issues"]:
    print("-", issue)

# Verify that the incorrect response was blocked
assert negative_release_result["released"] is False
assert negative_release_result["response"] is None
assert any(
    "Accuracy does not match evidence" in issue
    for issue in negative_release_result["issues"]
)

print("\nNegative Response Gate Test: PASSED")


Response Released: False
Validation Message: The response was blocked because it failed the validation checks.

Detected Issues:
- Accuracy does not match evidence.

Negative Response Gate Test: PASSED



### Audit Logging Integration

Audit logging records the execution history of the ModelGuard agent.

It stores information about user requests, execution plans, tool usage, reflection results, and final response status.

This provides traceability and supports later analysis of agent behavior, execution failures, and response validation decisions.

Before integration, the existing logging function is inspected to ensure that the required data is passed in the correct format.


In [80]:

print("Audit Logging Function Signature:")
print(inspect.signature(log_agent_execution))

print("\nAudit Logging Function Implementation:")
print(inspect.getsource(log_agent_execution))


Audit Logging Function Signature:
(user_request, planning_result, react_result, reflection_result, final_response)

Audit Logging Function Implementation:
def log_agent_execution(
    user_request,
    planning_result,
    react_result,
    reflection_result,
    final_response
):
    plan_data = planning_result.get("plan", [])
    tool_names = [
        step.get("tool_name")
        for step in plan_data
        if isinstance(step, dict)
    ]

    execution_records = [
        {
            "tool_name": item.get("tool_name"),
            "status": item.get("status"),
            "execution_time": item.get("execution_time")
        }
        for item in react_result.get("execution_log", [])
    ]

    reflection_data = (
        reflection_result.model_dump()
        if hasattr(reflection_result, "model_dump")
        else reflection_result
    )

    response_data = (
        final_response.model_dump()
        if hasattr(final_response, "model_dump")
        else final_response
    


### Improving Audit Logging

The audit logging function is updated to support both approved and blocked responses.

It also records the response validation decision and any detected issues. This ensures that failed validation attempts remain traceable without storing the full customer prediction data.


In [81]:

def log_agent_execution(
    user_request,
    planning_result,
    react_result,
    reflection_result,
    final_response,
    validation_result=None
):
    # Extract planned tools
    plan_data = planning_result.get("plan", [])

    tool_names = []
    for step in plan_data:
        if isinstance(step, dict):
            tool_names.append(step.get("tool_name"))
        elif hasattr(step, "model_dump"):
            tool_names.append(step.model_dump().get("tool_name"))

    # Extract tool execution records
    execution_records = [
        {
            "tool_name": item.get("tool_name"),
            "status": item.get("execution_status", item.get("status")),
            "execution_time": item.get("execution_time")
        }
        for item in react_result.get("execution_log", [])
    ]

    # Convert reflection result
    reflection_data = (
        reflection_result.model_dump()
        if hasattr(reflection_result, "model_dump")
        else reflection_result
    )

    # Handle approved and blocked responses
    if final_response is None:
        response_data = None
    elif hasattr(final_response, "model_dump"):
        response_data = final_response.model_dump()
    else:
        response_data = final_response

    # Extract validation information
    validation_data = validation_result or {}

    if response_data is not None:
        response_status = response_data.get("status", "unknown")
    elif validation_data.get("released") is False:
        response_status = "blocked"
    else:
        response_status = "not_generated"

    record = {
        "timestamp": datetime.now().astimezone().isoformat(),
        "agent": AGENT_NAME,
        "model": MODEL_NAME,
        "user_request": user_request,
        "planned_tools": tool_names,
        "execution_status": react_result.get("status"),
        "iterations": react_result.get("iterations"),
        "used_tools": react_result.get("used_tools", []),
        "tool_execution": execution_records,
        "reflection": reflection_data,
        "validation": {
            "released": validation_data.get("released"),
            "issues": validation_data.get("issues", [])
        },
        "final_response_status": response_status
    }

    with open(AUDIT_LOG_FILE, "a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")

    return record



### Audit Logging Integration Test

This test verifies that the updated logging function correctly records an approved response and a blocked response.

It also checks whether the validation decision and detected issues are stored in the audit log.


In [82]:

# Test logging an approved response
approved_log = log_agent_execution(
    user_request="Analyze customer churn and model performance.",
    planning_result={"plan": [{"tool_name": "predict_customer_churn"}]},
    react_result=combined_agent_result,
    reflection_result=reflection_result,
    final_response=release_result["response"],
    validation_result=release_result
)

print("Approved Response Log:")
print(json.dumps(approved_log, indent=2, ensure_ascii=False))


Approved Response Log:
{
  "timestamp": "2026-11-25T18:12:38.425185+03:00",
  "agent": "ModelGuard Agent",
  "model": "qwen2.5:1.5b",
  "user_request": "Analyze customer churn and model performance.",
  "planned_tools": [
    "predict_customer_churn"
  ],
  "execution_status": "completed",
  "iterations": 3,
  "used_tools": [
    "predict_customer_churn",
    "get_performance_baseline",
    "get_drift_analysis"
  ],
  "tool_execution": [
    {
      "tool_name": "predict_customer_churn",
      "status": "success",
      "execution_time": 0.0327
    },
    {
      "tool_name": "get_performance_baseline",
      "status": "success",
      "execution_time": 0.0008
    },
    {
      "tool_name": "get_drift_analysis",
      "status": "success",
      "execution_time": 0.19
    }
  ],
  "reflection": {
    "status": "approved",
    "evidence_supported": true,
    "identified_issues": [],
    "correction_required": false
  },
  "validation": {
    "released": true,
    "issues": []
  },
  "fi


### Blocked Response Logging Test

This test verifies that the audit logging function records responses rejected by the validation gate.

The log should preserve the blocked status and the validation issues without storing the rejected response as an approved final answer.


In [83]:

blocked_log = log_agent_execution(
    user_request="Analyze customer churn with incorrect performance metrics.",
    planning_result={"plan": [{"tool_name": "get_performance_baseline"}]},
    react_result=combined_agent_result,
    reflection_result=reflection_result,
    final_response=negative_release_result["response"],
    validation_result=negative_release_result
)

print("Blocked Response Log:")
print(json.dumps(blocked_log, indent=2, ensure_ascii=False))

assert blocked_log["validation"]["released"] is False
assert blocked_log["final_response_status"] == "blocked"
assert blocked_log["validation"]["issues"]

print("\nBlocked Response Logging Test: PASSED")


Blocked Response Log:
{
  "timestamp": "2026-11-25T18:12:38.441328+03:00",
  "agent": "ModelGuard Agent",
  "model": "qwen2.5:1.5b",
  "user_request": "Analyze customer churn with incorrect performance metrics.",
  "planned_tools": [
    "get_performance_baseline"
  ],
  "execution_status": "completed",
  "iterations": 3,
  "used_tools": [
    "predict_customer_churn",
    "get_performance_baseline",
    "get_drift_analysis"
  ],
  "tool_execution": [
    {
      "tool_name": "predict_customer_churn",
      "status": "success",
      "execution_time": 0.0327
    },
    {
      "tool_name": "get_performance_baseline",
      "status": "success",
      "execution_time": 0.0008
    },
    {
      "tool_name": "get_drift_analysis",
      "status": "success",
      "execution_time": 0.19
    }
  ],
  "reflection": {
    "status": "approved",
    "evidence_supported": true,
    "identified_issues": [],
    "correction_required": false
  },
  "validation": {
    "released": false,
    "issues"


### Audit Log Verification

This test reads the audit log file and verifies that both approved and blocked response records were saved correctly.


In [84]:

with open(AUDIT_LOG_FILE, "r", encoding="utf-8") as file:
    audit_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

print("Total Audit Records:", len(audit_records))

print("\nLatest Two Records:")
for record in audit_records[-2:]:
    print({
        "user_request": record["user_request"],
        "validation_released": record["validation"]["released"],
        "final_response_status": record["final_response_status"],
        "validation_issues": record["validation"]["issues"]
    })

assert audit_records[-2]["final_response_status"] == "success"
assert audit_records[-1]["final_response_status"] == "blocked"

print("\nAudit Log Verification: PASSED")


Total Audit Records: 36

Latest Two Records:
{'user_request': 'Analyze customer churn and model performance.', 'validation_released': True, 'final_response_status': 'success', 'validation_issues': []}
{'user_request': 'Analyze customer churn with incorrect performance metrics.', 'validation_released': False, 'final_response_status': 'blocked', 'validation_issues': ['Accuracy does not match evidence.']}

Audit Log Verification: PASSED



### Generating and Extracting the Execution Plan

The memory-aware planner generates a new execution plan based on the user's request.

The plan is then extracted and validated against the available ModelGuard tools.


In [85]:

# Generate a new planning result
planning_test = reason_and_plan_with_memory(
    "Analyze customer churn prediction, model performance, and feature drift."
)

# Extract the planning output
planning_output = planning_test["planning_result"]
execution_plan = planning_output["plan"]

# Display the execution plan
print("Planning Status:", planning_output["status"])
print("\nExecution Plan:")

for step in execution_plan:
    print(
        f"Step {step['step']}: "
        f"{step['tool_name']} - {step['purpose']}"
    )

# Validate planned tools
planned_tool_names = [
    step["tool_name"] for step in execution_plan
]

unknown_tools = [
    name for name in planned_tool_names
    if name not in TOOL_REGISTRY
]

assert planning_output["status"] == "success"
assert len(execution_plan) > 0
assert len(unknown_tools) == 0

print("\nPlanned Tools:", planned_tool_names)
print("Unknown Tools:", unknown_tools)
print("Execution Plan Validation: PASSED")


Planning Status: success

Execution Plan:
Step 1: get_performance_baseline - Retrieve the baseline performance metrics of the churn prediction model.
Step 2: get_drift_analysis - Analyze statistical data drift between reference and comparison datasets.

Planned Tools: ['get_performance_baseline', 'get_drift_analysis']
Unknown Tools: []
Execution Plan Validation: PASSED



### ReAct Execution Integration

The validated execution plan is passed to the ReAct agent.

The agent executes the selected tools, collects their outputs as evidence, and generates a structured execution result.

This step tests the connection between the planning and tool execution components.


In [86]:

# Execute the validated plan using the ReAct agent

react_integration_result = run_react_agent(
    user_request=planning_test["user_request"],
    execution_plan=execution_plan
)

print("ReAct Execution Status:", react_integration_result["status"])
print("\nUsed Tools:")
print(react_integration_result.get("used_tools", []))

print("\nEvidence Sources:")
print(list(react_integration_result.get("evidence", {}).keys()))

print("\nExecution Log:")
print(json.dumps(
    react_integration_result.get("execution_log", []),
    indent=2,
    ensure_ascii=False,
    default=str
))

assert react_integration_result["status"] == "completed"
assert set(planned_tool_names).issubset(
    set(react_integration_result.get("used_tools", []))
)

print("\nPlanner-to-ReAct Integration Test: PASSED")


ReAct Execution Status: completed

Used Tools:
['get_performance_baseline', 'get_drift_analysis']

Evidence Sources:
['get_performance_baseline', 'get_drift_analysis']

Execution Log:
[
  {
    "iteration": 1,
    "tool_name": "get_performance_baseline",
    "purpose": "Retrieve the baseline performance metrics of the churn prediction model.",
    "execution_status": "success",
    "execution_time": 0.0016
  },
  {
    "iteration": 2,
    "tool_name": "get_drift_analysis",
    "purpose": "Analyze statistical data drift between reference and comparison datasets.",
    "execution_status": "success",
    "execution_time": 0.2908
  }
]

Planner-to-ReAct Integration Test: PASSED



### ReAct to Structured Response Integration

The outputs collected by the ReAct agent are passed to the structured response builder.

The builder organizes the available evidence into key findings, evidence sources, and limitations.

This step verifies that the execution results can be converted into a structured response.


In [87]:

# Build a structured response from the ReAct execution results

integrated_response = build_structured_response(
    react_integration_result
)

print("Response Status:", integrated_response.status)

print("\nAnswer:")
print(integrated_response.answer)

print("\nKey Findings:")
for finding in integrated_response.key_findings:
    print("-", finding)

print("\nEvidence Sources:")
for source in integrated_response.evidence:
    print("-", source)

print("\nLimitations:")
for limitation in integrated_response.limitations:
    print("-", limitation)

# Validate the structured response
assert isinstance(integrated_response, FinalResponse)
assert integrated_response.status in ["success", "partial", "error"]
assert len(integrated_response.answer) > 0

print("\nReAct-to-Structured-Response Test: PASSED")


Response Status: success

Answer:

### Churn Prediction Model Assessment

**Model:** XGBoost

**Evaluation Dataset:** Held-out Test Dataset

**Classification Threshold:** 0.56

#### Baseline Performance Metrics

- **Accuracy:** 0.7644
- **Precision:** 0.5407
- **Recall:** 0.7460
- **F1-Score:** 0.6270
- **ROC-AUC:** 0.8463
- **PR-AUC:** 0.6622

#### Drift Analysis

**Analysis Type:** Reference vs Comparison Data

**Number of Analyzed Features:** 19

**Number of Drifted Features:** 0

No statistically significant drift was detected across the 19 analyzed features.

#### Limitations

The drift analysis compares reference and comparison
datasets. It does not represent live production monitoring.


Key Findings:
- Accuracy: 0.7643718949609652
- Precision: 0.5406976744186046
- Recall: 0.7459893048128342
- F1-Score: 0.6269662921348315
- ROC-AUC: 0.8462566844919786
- PR-AUC: 0.662242549619586
- Analyzed Features: 19
- Drifted Features: 0

Evidence Sources:
- Model performance baseline
- Featu


### Structured Response to Reflection Integration

The structured response is passed to the reflection component for evidence-based validation.

The reflection component checks whether the reported numerical findings are consistent with the outputs collected by the ReAct agent.

It also identifies missing or unsupported evidence and determines whether the response requires correction.


In [88]:

# Validate the structured response against ReAct evidence

integrated_reflection = reflect_on_response(
    integrated_response,
    react_integration_result
)

print("Reflection Status:", integrated_reflection.status)
print("Evidence Supported:", integrated_reflection.evidence_supported)
print("Correction Required:", integrated_reflection.correction_required)

print("\nIdentified Issues:")
for issue in integrated_reflection.identified_issues:
    print("-", issue)

assert integrated_reflection.status in [
    "approved", "needs_correction", "rejected"
]

print("\nStructured Response-to-Reflection Test: PASSED")


Reflection Status: approved
Evidence Supported: True
Correction Required: False

Identified Issues:

Structured Response-to-Reflection Test: PASSED



### Reflection to Response Validation Integration

The structured response and its supporting evidence are passed to the response validation gate.

The gate combines factual validation and reflection results to determine whether the response can be released to the user.

Responses that fail validation are blocked, while responses that pass all checks are approved for delivery.


In [89]:

# Validate and release the integrated response

integrated_release_result = release_agent_response(
    response=integrated_response,
    react_result=react_integration_result,
    evidence=react_integration_result["evidence"]
)

print("Response Released:", integrated_release_result["released"])
print("Validation Message:", integrated_release_result["message"])

print("\nValidation Issues:")
for issue in integrated_release_result["issues"]:
    print("-", issue)

# Verify the response gate
assert integrated_release_result["released"] is True
assert integrated_release_result["response"] is not None
assert len(integrated_release_result["issues"]) == 0

print("\nResponse Validation Integration Test: PASSED")


Response Released: True
Validation Message: The response passed validation and is approved for delivery.

Validation Issues:

Response Validation Integration Test: PASSED


### Integrated Workflow Audit Logging

The approved response, execution plan, ReAct results, and reflection output are passed to the audit logging function.

The test verifies that the complete workflow is recorded, including tool execution statuses and the final validation decision.

In [90]:

# Record the integrated agent execution

integrated_audit_record = log_agent_execution(
    user_request=planning_test["user_request"],
    planning_result=planning_test["planning_result"],
    react_result=react_integration_result,
    reflection_result=integrated_reflection,
    final_response=integrated_release_result["response"],
    validation_result=integrated_release_result
)

print("Integrated Audit Record:")
print(json.dumps(
    integrated_audit_record,
    indent=2,
    ensure_ascii=False,
    default=str
))

# Validate the audit record
assert integrated_audit_record["execution_status"] == "completed"
assert integrated_audit_record["validation"]["released"] is True
assert integrated_audit_record["final_response_status"] == "success"

assert all(
    item["status"] == "success"
    for item in integrated_audit_record["tool_execution"]
)

print("\nIntegrated Audit Logging Test: PASSED")


Integrated Audit Record:
{
  "timestamp": "2026-11-25T18:12:51.403052+03:00",
  "agent": "ModelGuard Agent",
  "model": "qwen2.5:1.5b",
  "user_request": "Analyze customer churn prediction, model performance, and feature drift.",
  "planned_tools": [
    "get_performance_baseline",
    "get_drift_analysis"
  ],
  "execution_status": "completed",
  "iterations": 2,
  "used_tools": [
    "get_performance_baseline",
    "get_drift_analysis"
  ],
  "tool_execution": [
    {
      "tool_name": "get_performance_baseline",
      "status": "success",
      "execution_time": 0.0016
    },
    {
      "tool_name": "get_drift_analysis",
      "status": "success",
      "execution_time": 0.2908
    }
  ],
  "reflection": {
    "status": "approved",
    "evidence_supported": true,
    "identified_issues": [],
    "correction_required": false
  },
  "validation": {
    "released": true,
    "issues": []
  },
  "final_response_status": "success"
}

Integrated Audit Logging Test: PASSED



### ModelGuard Orchestrator Preparation

This step checks the interfaces of the existing components before integrating them into one execution workflow.

In [91]:

import inspect

components = [
    reason_and_plan_with_memory,
    run_react_agent,
    build_structured_response,
    reflect_on_response,
    release_agent_response,
    log_agent_execution
]

for component in components:
    print(f"{component.__name__}: {inspect.signature(component)}")


reason_and_plan_with_memory: (user_request)
run_react_agent: (user_request, execution_plan, tool_inputs=None)
build_structured_response: (react_result)
reflect_on_response: (response_data, react_result)
release_agent_response: (response, react_result, evidence, required_evidence=None)
log_agent_execution: (user_request, planning_result, react_result, reflection_result, final_response, validation_result=None)



### ModelGuard Orchestrator Implementation

The orchestrator connects the existing ModelGuard components into one execution workflow.

It receives a user request, retrieves relevant memory, generates and validates an execution plan, executes the required tools, creates a structured response, performs reflection and response validation, and records the execution in the audit log.

Only responses that pass the validation gate are returned to the user.


In [92]:

def run_modelguard_agent(user_request, tool_inputs=None):

    planning_result = None
    react_result = None
    reflection_result = None
    structured_response = None
    release_result = None

    try:
        # 1. Memory and Planning
        planning_result = reason_and_plan_with_memory(user_request)

        planning_output = planning_result["planning_result"]
        execution_plan = planning_output["plan"]

        if planning_output["status"] != "success":
            raise ValueError("Planning was not successful.")

        # 2. Validate planned tools
        planned_tools = [step["tool_name"] for step in execution_plan]

        if not planned_tools:
            raise ValueError("The execution plan is empty.")

        unknown_tools = [
            name for name in planned_tools
            if name not in TOOL_REGISTRY
        ]

        if unknown_tools:
            raise ValueError(f"Unknown tools: {unknown_tools}")

        # 3. ReAct Execution
        react_result = run_react_agent(
            user_request=user_request,
            execution_plan=execution_plan,
            tool_inputs=tool_inputs
        )

        # 4. Structured Response
        structured_response = build_structured_response(react_result)

        # 5. Reflection
        reflection_result = reflect_on_response(
            structured_response,
            react_result
        )

        # 6. Response Validation Gate
        release_result = release_agent_response(
            response=structured_response,
            react_result=react_result,
            evidence=react_result.get("evidence", {})
        )

        # 7. Audit Logging
        audit_record = log_agent_execution(
            user_request=user_request,
            planning_result=planning_result["planning_result"],
            react_result=react_result,
            reflection_result=reflection_result,
            final_response=release_result["response"],
            validation_result=release_result
        )

        return {
            "status": "success" if release_result["released"] else "blocked",
            "response": release_result["response"],
            "reflection": reflection_result.model_dump(),
            "validation": release_result,
            "audit_record": audit_record
        }

    except Exception as error:
        return {
            "status": "error",
            "response": None,
            "error": str(error)
        }


print("ModelGuard Orchestrator defined successfully.")


ModelGuard Orchestrator defined successfully.


In [93]:

orchestrator_result = run_modelguard_agent(
    "Analyze customer churn model performance and feature drift."
)

print("System Status:", orchestrator_result["status"])

if orchestrator_result["status"] == "success":
    print("\nFinal Answer:")
    print(orchestrator_result["response"]["answer"])

    print("\nReflection:")
    print(orchestrator_result["reflection"])

    print("\nValidation:")
    print(orchestrator_result["validation"]["released"])

    print("\nAudit Logging:")
    print(orchestrator_result["audit_record"]["final_response_status"])

else:
    print("Error:", orchestrator_result.get("error"))


System Status: success

Final Answer:

### Churn Prediction Model Assessment

**Model:** XGBoost

**Evaluation Dataset:** Held-out Test Dataset

**Classification Threshold:** 0.56

#### Baseline Performance Metrics

- **Accuracy:** 0.7644
- **Precision:** 0.5407
- **Recall:** 0.7460
- **F1-Score:** 0.6270
- **ROC-AUC:** 0.8463
- **PR-AUC:** 0.6622

#### Drift Analysis

**Analysis Type:** Reference vs Comparison Data

**Number of Analyzed Features:** 19

**Number of Drifted Features:** 0

No statistically significant drift was detected across the 19 analyzed features.

#### Limitations

The drift analysis compares reference and comparison
datasets. It does not represent live production monitoring.


Reflection:
{'status': 'approved', 'evidence_supported': True, 'identified_issues': [], 'correction_required': False}

Validation:
True

Audit Logging:
success


## System Architecture

ModelGuard integrates a local language model, an agent runtime, and a set of model evaluation and governance tools. The architecture illustrates how user requests are processed, how tools provide evidence, and how the agent generates and validates its final response.

### Architecture Diagram

![ModelGuard System Architecture](ModelGuard_System_Architecture.svg)

### Architecture Interpretation

The system follows an agent-based workflow. The user interacts with the Streamlit interface, while the agent runtime coordinates the language model and available tools.

Tool outputs provide evidence for the final response. The reflection and validation stages review the generated answer before it is returned to the interface.

The architecture represents the intended workflow; the validation results assess whether the agent follows it correctly.

### Comprehensive Workflow Test

The comprehensive test evaluated the available system and governance capabilities. The agent successfully executed the selected ten tools and generated a response.

However, the response did not fully cover all requested governance components. Impact assessment and incident management were not included in the executed workflow.

**Result:** Tool execution passed, while governance coverage remained incomplete.

### Focused Governance Test

The focused test required the agent to address impact assessment, incident management, risk management, governance controls, and system status without repeating unrelated analyses.

The planner selected several previously used tools and omitted the required impact assessment and incident management tools. The reflection and validation stages did not identify these omissions.

**Result:** Task-scope adherence failed, and governance completeness was not demonstrated.

## Risk Prioritization

The identified risks were ranked using the probability–severity scoring approach. This prioritization supports the selection of governance actions according to their relative risk scores.

| Risk ID | Risk | Probability | Severity | Score |
|---|---|---:|---:|---:|
| R02 | Model Drift | 4 | 4 | 16 |
| R01 | Data Quality | 3 | 4 | 12 |
| R03 | Performance Degradation | 3 | 4 | 12 |
| R04 | Bias | 3 | 4 | 12 |
| R06 | Overreliance | 3 | 4 | 12 |
| R05 | Privacy | 2 | 5 | 10 |
| R07 | Unauthorized Access | 2 | 5 | 10 |

Risk Score = Probability × Severity

Model drift received the highest score and represents the primary risk priority in the current register. Privacy and unauthorized access also require attention because their severity scores are high, despite their lower probability estimates.

## AI Lifecycle Status

The following table summarizes the documented status of ModelGuard across its lifecycle stages.

| Lifecycle Stage | Status |
|---|---|
| Planning | Completed |
| Data Preparation | Completed |
| Development | Completed |
| Evaluation | Partially Completed |
| Deployment | Planned |
| Monitoring | Prototype Completed |
| Maintenance | Planned |
| Retirement | Planned |

The monitoring stage is based on the current prototype and stored data. It does not represent continuous production monitoring.

## Responsible AI and Governance References

ModelGuard considers responsible AI principles as part of its governance structure. These principles guide risk identification, control design, and system evaluation.